# Unlimited-OCR 2 (A100)

Drive-native PDF/image batch OCR with built-in /demo browser test UI; official Transformers backend, local SSD model and kernels, persistent Drive results, Caddy auth and asynchronous CLI jobs.

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "unlimited-ocr")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "8000"))
AUTH         = os.environ.get("CT_AUTH", "caddy")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "18080"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = os.environ.get("CT_WORK_DIR", "/content/ct")             # logs + runtime configs (0700)
CACHE_DIR = os.environ.get("CT_CACHE_DIR", "")
BIN_DIR = os.environ.get("CT_BIN_DIR", "/usr/local/bin")
os.makedirs(WORK, mode=0o700, exist_ok=True)
os.chmod(WORK, 0o700)
os.makedirs(BIN_DIR, exist_ok=True)
if BIN_DIR not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = BIN_DIR + os.pathsep + os.environ.get("PATH", "")
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to <WORK>/<name>.log (0600)."""
    lp = f"{WORK}/{name}.log"
    fd = os.open(lp, os.O_WRONLY | os.O_CREAT | os.O_APPEND, 0o600)
    os.chmod(lp, 0o600)
    with os.fdopen(fd, "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

CADDY_VERSION = "2.10.2"

def ct_write_private_text(path, content):
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    os.chmod(path, 0o600)
    with os.fdopen(fd, "w", encoding="utf-8") as handle:
        handle.write(content)
        handle.flush()
        os.fsync(handle.fileno())

def ct_proc_start_time(pid):
    try:
        fields = open(f"/proc/{int(pid)}/stat", "r", encoding="utf-8", errors="replace").read().rsplit(")", 1)[1].split()
        if fields[0] == "Z":
            return None
        return fields[19]
    except (OSError, IndexError, ValueError):
        return None

def ct_stop_marked_proc(name, timeout=10):
    import signal
    marker = f"{WORK}/{name}.proc.json"
    if not os.path.exists(marker):
        return
    try:
        info = json.loads(open(marker, "r", encoding="utf-8").read())
        pid = int(info.get("pid") or 0)
        start_time = info.get("start_time")
    except (OSError, ValueError, TypeError):
        pid, start_time = 0, None
    if pid > 0 and start_time and ct_proc_start_time(pid) == start_time:
        for sig in (signal.SIGTERM, signal.SIGKILL):
            if ct_proc_start_time(pid) != start_time:
                break
            try:
                os.killpg(os.getpgid(pid), sig)
            except (ProcessLookupError, PermissionError, OSError):
                try:
                    os.kill(pid, sig)
                except (ProcessLookupError, OSError):
                    break
            deadline = time.monotonic() + timeout
            while time.monotonic() < deadline:
                if ct_proc_start_time(pid) != start_time:
                    break
                time.sleep(0.1)
    try:
        os.unlink(marker)
    except OSError:
        pass

def ct_spawn_marked(name, args, env=None, cwd=None):
    ct_stop_marked_proc(name)
    proc = ct_spawn(name, args, env=env, cwd=cwd)
    marker = f"{WORK}/{name}.proc.json"
    ct_write_private_text(marker, json.dumps({"pid": proc.pid, "start_time": ct_proc_start_time(proc.pid)}))
    return proc

def _ct_file_sha256(path):
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(65536)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def ct_ensure_caddy():
    import hashlib, platform, tarfile, tempfile
    arch = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")
    dest = f"{BIN_DIR}/caddy"
    status_label = "miss"
    manifest = f"{CACHE_DIR}/manifests/caddy.json" if CACHE_DIR else ""
    if os.path.exists(dest):
        if not manifest:
            return dest
        try:
            meta = json.loads(open(manifest, "r", encoding="utf-8").read())
            if meta.get("artifact") == "caddy" and meta.get("version") == CADDY_VERSION and meta.get("sha256") == _ct_file_sha256(dest):
                print("CT_CACHE_EVENT artifact=caddy status=hit", flush=True)
                return dest
        except (OSError, ValueError):
            pass
        try:
            os.unlink(dest)
        except OSError:
            pass
        status_label = "corrupt_rebuilt"
    asset = f"caddy_{CADDY_VERSION}_linux_{arch}.tar.gz"
    checks_asset = f"caddy_{CADDY_VERSION}_checksums.txt"
    base = f"https://github.com/caddyserver/caddy/releases/download/v{CADDY_VERSION}/"
    os.makedirs(BIN_DIR, exist_ok=True)
    with tempfile.TemporaryDirectory() as tmp:
        tgz = os.path.join(tmp, asset)
        with urllib.request.urlopen(base + asset, timeout=60) as src, open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        checks = urllib.request.urlopen(base + checks_asset, timeout=60).read().decode("utf-8")
        expected = next((line.split()[0] for line in checks.splitlines() if line.split() and line.split()[-1].lstrip("*") == asset), None)
        algo = {64: hashlib.sha256, 128: hashlib.sha512}.get(len(expected or ""))
        if algo is None or algo(open(tgz, "rb").read()).hexdigest() != expected:
            raise RuntimeError(f"Caddy checksum verification failed for {asset}")
        partial = f"{dest}.{os.getpid()}.partial"
        try:
            with tarfile.open(tgz) as tar:
                entry = tar.getmember("caddy")
                with tar.extractfile(entry) as src, open(partial, "wb") as out:
                    shutil.copyfileobj(src, out)
            os.chmod(partial, 0o755)
            os.replace(partial, dest)
        finally:
            if os.path.exists(partial):
                os.unlink(partial)
    if manifest:
        os.makedirs(os.path.dirname(manifest), exist_ok=True)
        tmp_m = f"{manifest}.{os.getpid()}.tmp"
        with open(tmp_m, "w", encoding="utf-8") as mf:
            mf.write(json.dumps({
                "artifact": "caddy",
                "version": CADDY_VERSION,
                "sha256": _ct_file_sha256(dest),
                "bin_path": dest,
                "verified_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }, indent=2) + "\n")
        os.replace(tmp_m, manifest)
        print(f"CT_CACHE_EVENT artifact=caddy status={status_label}", flush=True)
    return dest

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None, heartbeat_interval=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    last_heartbeat = started
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        now_m = time.monotonic()
        if heartbeat is not None and (heartbeat_interval is None or (now_m - last_heartbeat) >= heartbeat_interval):
            last_heartbeat = now_m
            try:
                heartbeat(int(now_m - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
OCR_CACHE_SOURCE = '"""Stage optional compiler caches using one Drive archive per environment/GPU."""\nimport hashlib\nimport os\nimport shutil\nimport tarfile\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\n\n\ndef _progress(log, message, operation):\n    stopped = threading.Event()\n    started = time.monotonic()\n\n    def heartbeat():\n        while not stopped.wait(10):\n            log(f\'{message}，已用时 {time.monotonic()-started:.0f} 秒\')\n\n    log(message)\n    thread = threading.Thread(target=heartbeat, daemon=True)\n    thread.start()\n    try:\n        return operation()\n    finally:\n        stopped.set()\n        thread.join()\n\n\ndef _files(root):\n    return sorted(p for p in root.rglob(\'*\') if p.is_file() and not p.is_symlink()\n                  and not p.name.endswith((\'.lock\', \'.tmp\')))\n\n\ndef _fingerprint(root, files):\n    digest = hashlib.sha256()\n    for path in files:\n        stat = path.stat()\n        digest.update(f\'{path.relative_to(root)}:{stat.st_size}:{stat.st_mtime_ns}\\n\'.encode())\n    return digest.hexdigest()\n\n\ndef _unpack(archive, local):\n    # Validate the whole archive before extracting; compiler caches contain\n    # regular files/directories only. Never follow archive or local symlinks.\n    with tarfile.open(archive, \'r:\') as tar:\n        members = tar.getmembers()\n        root = local.resolve()\n        for member in members:\n            destination = local / member.name\n            if (not (member.isfile() or member.isdir()) or Path(member.name).is_absolute()\n                    or not destination.resolve().is_relative_to(root)):\n                raise ValueError(\'缓存归档包含不安全的路径或文件类型\')\n            current = destination\n            while current != local:\n                if current.is_symlink():\n                    raise ValueError(\'缓存目标包含符号链接\')\n                current = current.parent\n        tar.extractall(local, members=members, filter=\'data\')\n\n\ndef restore_cache(local, legacy, archive_dir, key, log=print):\n    local, legacy, archive_dir = map(Path, (local, legacy, archive_dir))\n    local.mkdir(parents=True, exist_ok=True)\n    archive = archive_dir / key / (local.name + \'.tar\')\n    stamp = local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')\n    if archive.is_file():\n        stat = archive.stat()\n        signature = f\'{stat.st_size}:{stat.st_mtime_ns}\'\n        if stamp.exists() and stamp.read_text() == signature and _files(local):\n            log(f\'{local.name}: 已在本地恢复，跳过重复传输\')\n            return \'local\'\n        try:\n            with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-restore-\') as tmp:\n                staged = Path(tmp) / \'cache.tar\'\n                _progress(log, f\'{local.name}: 从网盘读取归档（{stat.st_size/1e6:.1f} MB）\',\n                          lambda: shutil.copyfile(archive, staged))\n                _progress(log, f\'{local.name}: 在本地 SSD 解包\', lambda: _unpack(staged, local))\n            stamp.write_text(signature)\n            (local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')).write_text(\n                _fingerprint(local, _files(local)))\n            log(f\'{local.name}: 归档恢复完成\')\n            return \'archive\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: 归档不可用，回退旧缓存（{error}）\')\n    if legacy.is_dir():\n        try:\n            def migrate():\n                shutil.copytree(legacy, local, dirs_exist_ok=True, symlinks=True)\n            _progress(log, f\'{local.name}: 首次迁移旧版网盘缓存（小文件较多时耗时较长）\', migrate)\n            if _files(local):\n                save_cache(local, archive_dir, key, log)\n                return \'legacy\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: 旧缓存迁移失败，回退按需生成（{error}）\')\n    log(f\'{local.name}: 无持久缓存，将由服务按需生成\')\n    return \'miss\'\n\n\ndef save_cache(local, archive_dir, key, log=print):\n    local, archive_dir = Path(local), Path(archive_dir)\n    files = _files(local)\n    if not files:\n        log(f\'{local.name}: 无缓存文件，跳过归档\')\n        return False\n    fingerprint = _fingerprint(local, files)\n    saved = local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')\n    destination = archive_dir / key / (local.name + \'.tar\')\n    if saved.exists() and saved.read_text() == fingerprint and destination.is_file():\n        log(f\'{local.name}: 内容未变化，跳过上传\')\n        return False\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_name(destination.name + \'.\' + uuid.uuid4().hex + \'.tmp\')\n    try:\n        with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-save-\') as tmp:\n            staged = Path(tmp) / \'cache.tar\'\n            def pack():\n                with tarfile.open(staged, \'w:\') as tar:\n                    for path in files:\n                        tar.add(path, arcname=str(path.relative_to(local)), recursive=False)\n            _progress(log, f\'{local.name}: 本地打包 {len(files)} 个文件\', pack)\n            _progress(log, f\'{local.name}: 保存归档到网盘（{staged.stat().st_size/1e6:.1f} MB）\',\n                      lambda: shutil.copyfile(staged, temporary))\n            os.replace(temporary, destination)\n        saved.write_text(fingerprint)\n        stat = destination.stat()\n        (local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')).write_text(\n            f\'{stat.st_size}:{stat.st_mtime_ns}\')\n        log(f\'{local.name}: 网盘归档已更新\')\n        return True\n    finally:\n        temporary.unlink(missing_ok=True)\n'
# @title 1. Setup -- pinned official Transformers backend; model on local SSD
from pathlib import Path

OCR_WORK = Path('/content/unlimited-ocr')
OCR_WORK.mkdir(exist_ok=True)
if not shutil.which('uv'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
OCR_PY = str(OCR_WORK / 'venv/bin/python')
if not Path(OCR_PY).exists():
    subprocess.run(['uv', 'venv', '--python', '3.10', str(OCR_WORK / 'venv')], check=True)
OCR_PACKAGES = ['torch==2.13.0', 'torchvision', 'pillow', 'transformers==4.52.1',
                'huggingface-hub==0.36.0', 'tokenizers==0.21.4', 'addict==2.4.0',
                'einops==0.8.2', 'easydict==1.13', 'pymupdf==1.27.2.2', 'requests']
OCR_MARKER = OCR_WORK / '.installed'
OCR_SIGNATURE = json.dumps(OCR_PACKAGES)
if not OCR_MARKER.exists() or OCR_MARKER.read_text() != OCR_SIGNATURE:
    subprocess.run(['uv', 'pip', 'install', '--python', OCR_PY, *OCR_PACKAGES], check=True)
    OCR_MARKER.write_text(OCR_SIGNATURE)
OCR_MODEL = OCR_WORK / 'model'
subprocess.run([OCR_PY, '-c', '''from huggingface_hub import snapshot_download
snapshot_download('baidu/Unlimited-OCR-2', revision='0480966938291445591f69f08a13b041c073a96a',
                  local_dir='/content/unlimited-ocr/model',
                  ignore_patterns=['assets/*', 'wheel/*', '*.pdf', '*.md'])
'''], check=True, env={**os.environ, 'HF_XET_CHUNK_CACHE_SIZE_BYTES': '0'})
shutil.rmtree('/root/.cache/huggingface/xet', ignore_errors=True)
OCR_POSTPROCESS = OCR_WORK / 'process_unlimitedOCR2.py'
if not OCR_POSTPROCESS.exists():
    with urllib.request.urlopen(
        'https://raw.githubusercontent.com/baidu/Unlimited-OCR/'
        'ea42e6d15ff93fdd0e8041fc0ac70880513453d3/utils/process_unlimitedOCR2.py',
        timeout=60,
    ) as _src, open(OCR_POSTPROCESS, 'wb') as _dst:
        shutil.copyfileobj(_src, _dst)
# Triton / Inductor compile on SSD; restore and persist caches by GPU and environment.
(OCR_WORK / 'colab_cache.py').write_text(OCR_CACHE_SOURCE)
sys.path.insert(0, str(OCR_WORK))
from colab_cache import restore_cache, save_cache
try:
    OCR_GPU = (
        subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True)
        .splitlines()[0]
        .strip()
        .replace(' ', '_')
    )
except (OSError, subprocess.SubprocessError, IndexError):
    OCR_GPU = 'cpu'
OCR_CACHE_KEY = 'torch213-uocr2-04809669-' + OCR_GPU
OCR_CACHE_PAIRS = []
for name in ('triton-cache', 'inductor-cache'):
    local = OCR_WORK / name
    try:
        restore_cache(local, Path(DATA_DIR) / name, Path(DATA_DIR) / 'kernel-archives', OCR_CACHE_KEY)
    except (OSError, ValueError) as _cache_err:
        print(f'Warning: cache restore skipped for {name}: {_cache_err}', flush=True)
    OCR_CACHE_PAIRS.append(local)
print('Unlimited-OCR 2 setup complete', flush=True)


In [ ]:
# @title 2. Unlimited-OCR Drive job service
(OCR_WORK / 'ocr_parameters.py').write_text('"""Public inference settings for the pinned Unlimited-OCR 2 Transformers model.\n\nOnly tokenizer, image_file(s), and output_path are owned by the job runner.\n"""\nimport argparse\nimport json\nimport math\n\n# name -> (JSON type, minimum, nullable). No upper caps beyond the model itself.\nPARAMETERS = {\n    \'prompt\': (str, None, False),\n    \'base_size\': (int, 1, False),\n    \'image_size\': (int, 1, False),\n    \'crop_mode\': (bool, None, True),\n    \'test_compress\': (bool, None, False),\n    \'save_results\': (bool, None, False),\n    \'eval_mode\': (bool, None, False),\n    \'max_length\': (int, 1, False),\n    \'max_new_tokens\': (int, 1, True),\n    \'tps_interval\': (int, 0, False),\n    \'no_repeat_ngram_size\': (int, 0, False),\n    \'ngram_window\': (int, 0, False),\n    \'ngram_whitelist_token_ids\': (list, None, True),\n    \'temperature\': (float, 0, False),\n    \'page_count_guard\': (bool, None, False),\n    \'max_page_tokens\': (int, None, False),\n    \'dash_topm\': (int, 1, False),\n    \'max_crops\': (int, 1, False),\n    \'return_dict\': (bool, None, False),\n}\n\n\ndef validate_parameters(value):\n    if not isinstance(value, dict):\n        raise ValueError(\'inference_params 必须是 JSON 对象\')\n    for name, item in value.items():\n        if name not in PARAMETERS:\n            raise ValueError(f\'不支持的推理参数: {name}；可用: {", ".join(PARAMETERS)}\')\n        kind, minimum, nullable = PARAMETERS[name]\n        if item is None and nullable:\n            continue\n        valid = type(item) is kind\n        if kind is float:\n            valid = type(item) in (int, float) and math.isfinite(item)\n        if not valid or (minimum is not None and item < minimum):\n            raise ValueError(f\'{name}: 类型必须是 {kind.__name__}\' + (f\'，值 >= {minimum}\' if minimum is not None else \'\'))\n        if kind is list and any(type(n) is not int or n < 0 for n in item):\n            raise ValueError(f\'{name} 必须是非负整数数组或 null\')\n    return dict(value)\n\n\ndef inference_kwargs(params, page_count, tokens_per_page):\n    """Keep historical service defaults; explicit values, including null, always win."""\n    result = dict(eval_mode=True, crop_mode=False, return_dict=True,\n                  max_new_tokens=min(120000, tokens_per_page * page_count),\n                  no_repeat_ngram_size=35, ngram_window=1024 if page_count > 1 else 128)\n    result.update(validate_parameters(params))\n    return result\n\n\ndef add_inference_arguments(parser):\n    parser.add_argument(\'--params-file\', help=\'本地 JSON 推理参数文件（使用官方参数名）\')\n    parser.add_argument(\'--params-json\', help=\'JSON 推理参数对象，覆盖参数文件\')\n    parser.add_argument(\'--param\', action=\'append\', default=[], metavar=\'KEY=JSON\',\n                        help=\'可重复；覆盖 JSON 配置，如 --param crop_mode=null\')\n    for name, (kind, _, _) in PARAMETERS.items():\n        flag = \'--\' + name.replace(\'_\', \'-\')\n        if kind is bool:\n            parser.add_argument(flag, dest=name, action=argparse.BooleanOptionalAction, default=None)\n        elif kind is list:\n            parser.add_argument(flag, dest=name, type=json.loads, default=None, metavar=\'JSON_ARRAY\')\n        else:\n            parser.add_argument(flag, dest=name, type=kind, default=None)\n\n\ndef cli_parameters(args):\n    from pathlib import Path\n    result = {}\n    for raw in ([Path(args.params_file).expanduser().read_text(encoding=\'utf-8\')] if args.params_file else []) + ([args.params_json] if args.params_json else []):\n        result.update(validate_parameters(json.loads(raw)))\n    for name in PARAMETERS:\n        value = getattr(args, name)\n        if value is not None:\n            result[name] = value\n    for setting in args.param:\n        name, separator, raw = setting.partition(\'=\')\n        if not separator:\n            raise ValueError(\'--param 格式必须为 KEY=JSON\')\n        result[name] = json.loads(raw)\n    return validate_parameters(result)\n', encoding='utf-8')
(OCR_WORK / 'unlimited_ocr_demo.py').write_text('"""Self-contained Unlimited-OCR 2 interactive test/demo Web UI and raw layout parser."""\nfrom __future__ import annotations\n\nimport re\n\nPAGE_SPLIT_RE = re.compile(r\'<\\s*/?\\s*PAGE\\s*\\d*\\s*>\', re.I)\nMERGE_RE = re.compile(r\'<\\s*/?\\s*MERGE\\s*>\', re.I)\nSTRAY_TOKEN_RE = re.compile(\n    r\'<[|｜]/?[a-zA-Z_]+[|｜]>|<image>\'\n    r\'|<\\s*/?\\s*PAGE\\s*\\d*\\s*>|<\\s*/?\\s*BLOCK\\s*>|<\\s*/?\\s*FORK\\s*>\'\n    r\'|<\\s*/?\\s*LEVEL\\s*\\d*\\s*[>}]?|<\\s*/?\\s*MERGE\\s*>\',\n    re.I,\n)\nBLOCK_RE = re.compile(\n    r\'<\\s*BLOCK\\s*>\\s*([a-zA-Z_]+)\'\n    r\'(?:\'\n    r\'\\s*<\\s*/?\\s*LEVEL\\s*(\\d*)\\s*[>}]?(?:\\s*\\[\\s*(-?\\d+)\\s*[,，]\\s*(-?\\d+)\\s*[,，]\\s*(-?\\d+)\\s*[,，]\\s*(-?\\d+)\'\n    r\'(?:\\s*[,，]\\s*-?\\d+)*\\s*\\])?\'\n    r\'|\'\n    r\'\\s*\\[\\s*(-?\\d+)\\s*[,，]\\s*(-?\\d+)\\s*[,，]\\s*(-?\\d+)\\s*[,，]\\s*(-?\\d+)\'\n    r\'(?:\\s*[,，]\\s*-?\\d+)*\\s*\\](?:\\s*<\\s*/?\\s*LEVEL\\s*(\\d*)\\s*[>}]?)?\'\n    r\')?\'\n    r\'\\s*(?:<\\s*FORK\\s*>)?\',\n    re.I,\n)\n_BASE_LABELS = {\n    \'text\', \'title\', \'header\', \'footer\', \'page_number\', \'image\', \'figure\',\n    \'chart\', \'seal\', \'table\', \'image_caption\', \'image_footnote\',\n    \'page_footnote\', \'equation\', \'ref_text\', \'aside_text\', \'algorithm\',\n}\n_BARE_LABELS = _BASE_LABELS | {f\'{p}{l}\' for p in (\'cross_\', \'append_\') for l in _BASE_LABELS}\nBARE_BLOCK_RE = re.compile(\n    r\'(?:(?<=<FORK>)|(?<=<PAGE>)|(?<=\\n)|^)\'\n    r\'(?:<[|｜]\\s*det\\s*[|｜]>)?\'\n    r\'((?:[a-zA-Z_][a-zA-Z_0-9]*)(?:\\s*<\\s*/?\\s*LEVEL\\s*\\d*\\s*[>}]?)?\\s*\'\n    r\'\\[\\s*-?\\d+(?:\\s*[,，]\\s*-?\\d+){3,}[^\\]]*\\])\',\n    re.I,\n)\n\n\ndef _scale(value: int) -> int:\n    return max(0, min(1000, int(value)))\n\n\ndef _base_label(label: str) -> str:\n    for prefix in (\'append_\', \'cross_\'):\n        if label.startswith(prefix):\n            return label[len(prefix):]\n    return label\n\n\ndef _fix_bare_blocks(text: str) -> str:\n    def _repl(match: re.Match[str]) -> str:\n        name = re.match(r\'[a-zA-Z_][a-zA-Z_0-9]*\', match.group(1)).group(0)\n        return f\'<BLOCK>{match.group(1)}\' if name.lower() in _BARE_LABELS else match.group(0)\n\n    fixed = BARE_BLOCK_RE.sub(_repl, text)\n    return re.sub(r\'\\n\\s*<FORK>\\s*(?=<BLOCK>)\', \'\\n\', fixed, flags=re.I)\n\n\ndef parse_raw_blocks(raw_text: str) -> list[dict]:\n    """Parse Unlimited-OCR 2 raw block output into normalized 0..1000 layout boxes."""\n    if not isinstance(raw_text, str) or not raw_text.strip():\n        return []\n    text = _fix_bare_blocks(raw_text)\n    segments = PAGE_SPLIT_RE.split(text)\n    if segments and not segments[0].strip():\n        segments = segments[1:]\n    blocks: list[dict] = []\n    for page_idx, seg in enumerate(segments, start=1):\n        if not seg.strip():\n            continue\n        matches = list(BLOCK_RE.finditer(seg))\n        if not matches:\n            cleaned = STRAY_TOKEN_RE.sub(\'\', MERGE_RE.sub(\'\', seg)).strip()\n            if cleaned:\n                blocks.append({\n                    \'index\': len(blocks) + 1,\n                    \'page\': page_idx,\n                    \'label\': \'__raw__\',\n                    \'base_label\': \'__raw__\',\n                    \'level\': None,\n                    \'bbox\': None,\n                    \'content\': cleaned,\n                })\n            continue\n        for idx, m in enumerate(matches):\n            if idx == 0:\n                pre = STRAY_TOKEN_RE.sub(\'\', MERGE_RE.sub(\'\', seg[:m.start()])).strip()\n                if pre:\n                    blocks.append({\n                        \'index\': len(blocks) + 1,\n                        \'page\': page_idx,\n                        \'label\': \'__raw__\',\n                        \'base_label\': \'__raw__\',\n                        \'level\': None,\n                        \'bbox\': None,\n                        \'content\': pre,\n                    })\n            end = matches[idx + 1].start() if idx + 1 < len(matches) else len(seg)\n            content = STRAY_TOKEN_RE.sub(\'\', MERGE_RE.sub(\'\', seg[m.end():end])).strip()\n            if m.group(2) is not None:\n                level = int(m.group(2)) if m.group(2) else None\n                bbox = ([_scale(m.group(k)) for k in range(3, 7)] if m.group(3) is not None else None)\n            else:\n                bbox = ([_scale(m.group(k)) for k in range(7, 11)] if m.group(7) is not None else None)\n                level = int(m.group(11)) if m.group(11) else None\n            label = m.group(1).lower()\n            blocks.append({\n                \'index\': len(blocks) + 1,\n                \'page\': page_idx,\n                \'label\': label,\n                \'base_label\': _base_label(label),\n                \'level\': level,\n                \'bbox\': bbox,\n                \'content\': content,\n            })\n    return blocks\n\n\ndef make_csp(nonce: str) -> str:\n    """Return a strict per-request CSP header without external origins."""\n    return (\n        "default-src \'self\'; "\n        f"script-src \'nonce-{nonce}\'; "\n        f"style-src \'nonce-{nonce}\'; "\n        "img-src \'self\' data: blob:; "\n        "connect-src \'self\'; "\n        "font-src \'self\'; "\n        "object-src \'none\'; "\n        "frame-ancestors \'none\'; "\n        "base-uri \'none\'"\n    )\n\n\nHTML_TEMPLATE = r"""<!doctype html>\n<html lang="zh-CN">\n<head>\n  <meta charset="utf-8">\n  <meta name="viewport" content="width=device-width, initial-scale=1">\n  <meta name="color-scheme" content="light dark">\n  <title>Unlimited-OCR 2 Online Demo</title>\n  <style nonce="__NONCE__">\n    :root {\n      --bg-light: #f4f6fa;\n      --bg-dark: #0d121a;\n      --surface-light: #ffffff;\n      --surface-dark: #151c27;\n      --surface-2-light: #eef2f8;\n      --surface-2-dark: #1d2635;\n      --text-light: #16202e;\n      --text-dark: #e6edf7;\n      --muted-light: #556377;\n      --muted-dark: #97a6bc;\n      --border-light: #d2dae6;\n      --border-dark: #2c384b;\n      --brand-light: #c86300;\n      --brand-dark: #f59e0b;\n      --brand-text-light: #ffffff;\n      --brand-text-dark: #1b1102;\n      --ok-light: #15803d;\n      --ok-dark: #86efac;\n      --ok-bg-light: #eafaf0;\n      --ok-bg-dark: #122d1f;\n      --danger-light: #b91c1c;\n      --danger-dark: #fca5a5;\n      --danger-bg-light: #fef2f2;\n      --danger-bg-dark: #3a1418;\n      --scroll-track-light: #e7ecf4;\n      --scroll-track-dark: #18202c;\n      --scroll-thumb-light: #69778b;\n      --scroll-thumb-dark: #8594aa;\n\n      --bg: var(--bg-light);\n      --surface: var(--surface-light);\n      --surface-2: var(--surface-2-light);\n      --text: var(--text-light);\n      --muted: var(--muted-light);\n      --border: var(--border-light);\n      --brand: var(--brand-light);\n      --brand-text: var(--brand-text-light);\n      --ok: var(--ok-light);\n      --ok-bg: var(--ok-bg-light);\n      --danger: var(--danger-light);\n      --danger-bg: var(--danger-bg-light);\n      --scroll-track: var(--scroll-track-light);\n      --scroll-thumb: var(--scroll-thumb-light);\n\n      @media (prefers-color-scheme: dark) {\n        --bg: var(--bg-dark);\n        --surface: var(--surface-dark);\n        --surface-2: var(--surface-2-dark);\n        --text: var(--text-dark);\n        --muted: var(--muted-dark);\n        --border: var(--border-dark);\n        --brand: var(--brand-dark);\n        --brand-text: var(--brand-text-dark);\n        --ok: var(--ok-dark);\n        --ok-bg: var(--ok-bg-dark);\n        --danger: var(--danger-dark);\n        --danger-bg: var(--danger-bg-dark);\n        --scroll-track: var(--scroll-track-dark);\n        --scroll-thumb: var(--scroll-thumb-dark);\n      }\n\n      @supports (color: light-dark(white, black)) {\n        --bg: light-dark(var(--bg-light), var(--bg-dark));\n        --surface: light-dark(var(--surface-light), var(--surface-dark));\n        --surface-2: light-dark(var(--surface-2-light), var(--surface-2-dark));\n        --text: light-dark(var(--text-light), var(--text-dark));\n        --muted: light-dark(var(--muted-light), var(--muted-dark));\n        --border: light-dark(var(--border-light), var(--border-dark));\n        --brand: light-dark(var(--brand-light), var(--brand-dark));\n        --brand-text: light-dark(var(--brand-text-light), var(--brand-text-dark));\n        --ok: light-dark(var(--ok-light), var(--ok-dark));\n        --ok-bg: light-dark(var(--ok-bg-light), var(--ok-bg-dark));\n        --danger: light-dark(var(--danger-light), var(--danger-dark));\n        --danger-bg: light-dark(var(--danger-bg-light), var(--danger-bg-dark));\n        --scroll-track: light-dark(var(--scroll-track-light), var(--scroll-track-dark));\n        --scroll-thumb: light-dark(var(--scroll-thumb-light), var(--scroll-thumb-dark));\n      }\n\n      --left-ratio: 50%;\n      color-scheme: light dark;\n      accent-color: var(--brand);\n      scrollbar-color: var(--scroll-thumb) var(--scroll-track);\n    }\n\n    @supports not (scrollbar-color: auto) {\n      .scroller { scrollbar-width: auto; }\n      .scroller::-webkit-scrollbar { width: 10px; height: 10px; }\n      .scroller::-webkit-scrollbar-thumb { background: var(--scroll-thumb); border-radius: 999px; }\n      .scroller::-webkit-scrollbar-track { background: var(--scroll-track); }\n    }\n\n    *, *::before, *::after { box-sizing: border-box; }\n    html, body { height: 100%; margin: 0; }\n    body {\n      font-family: system-ui, -apple-system, "Segoe UI", Roboto, sans-serif;\n      line-height: 1.5;\n      background: var(--bg);\n      color: var(--text);\n    }\n\n    .visually-hidden:where(:not(:focus-within, :active)) {\n      position: absolute !important;\n      clip-path: inset(50%) !important;\n      overflow: hidden !important;\n      width: 1px !important;\n      height: 1px !important;\n      margin: -1px !important;\n      padding: 0 !important;\n      border: 0 !important;\n      white-space: nowrap !important;\n    }\n\n    .app-shell {\n      display: grid;\n      grid-template-rows: auto 1fr;\n      height: 100dvh;\n      min-height: 540px;\n    }\n\n    .app-header {\n      display: flex;\n      flex-wrap: wrap;\n      align-items: center;\n      justify-content: space-between;\n      gap: 0.75rem;\n      padding: 0.7rem 1rem;\n      background: var(--surface);\n      border-block-end: 1px solid var(--border);\n    }\n\n    .brand {\n      display: flex;\n      align-items: center;\n      gap: 0.65rem;\n      min-width: 0;\n    }\n    .brand h1 {\n      margin: 0;\n      font-size: 1.05rem;\n      font-weight: 700;\n      letter-spacing: -0.01em;\n    }\n    .brand-accent { color: var(--brand); }\n    .status-pill {\n      display: inline-flex;\n      align-items: center;\n      gap: 0.35rem;\n      padding: 0.15rem 0.55rem;\n      border-radius: 999px;\n      font-size: 0.78rem;\n      font-weight: 600;\n      background: var(--ok-bg);\n      color: var(--ok);\n      border: 1px solid color-mix(in srgb, var(--ok) 35%, transparent);\n    }\n    .status-pill.is-error {\n      background: var(--danger-bg);\n      color: var(--danger);\n      border-color: color-mix(in srgb, var(--danger) 35%, transparent);\n    }\n\n    .header-actions {\n      display: flex;\n      flex-wrap: wrap;\n      align-items: center;\n      gap: 0.45rem;\n    }\n    button, .btn {\n      font: inherit;\n      font-size: 0.85rem;\n      font-weight: 600;\n      padding: 0.42rem 0.78rem;\n      border-radius: 0.5rem;\n      border: 1px solid var(--border);\n      background: var(--surface-2);\n      color: var(--text);\n      cursor: pointer;\n    }\n    button:hover:not(:disabled) {\n      border-color: var(--brand);\n    }\n    button.primary {\n      background: var(--brand);\n      color: var(--brand-text);\n      border-color: var(--brand);\n    }\n    button:disabled {\n      opacity: 0.48;\n      cursor: not-allowed;\n    }\n\n    .workspace {\n      display: grid;\n      grid-template-columns: 18.5rem minmax(0, 1fr);\n      min-height: 0;\n    }\n\n    .config-panel {\n      display: flex;\n      flex-direction: column;\n      min-height: 0;\n      min-width: 0;\n      background: var(--surface);\n      border-inline-end: 1px solid var(--border);\n    }\n\n    .panel-header {\n      display: flex;\n      align-items: center;\n      justify-content: space-between;\n      gap: 0.5rem;\n      padding: 0.5rem 0.8rem;\n      min-height: 2.65rem;\n      background: var(--surface-2);\n      border-block-end: 1px solid var(--border);\n      font-size: 0.85rem;\n      font-weight: 700;\n    }\n\n    .config-scroll {\n      flex: 1;\n      overflow-x: hidden;\n      overflow-y: auto;\n      padding: 0.85rem;\n      display: grid;\n      grid-template-columns: minmax(0, 1fr);\n      gap: 1rem;\n      align-content: start;\n    }\n\n    .section-title {\n      margin: 0 0 0.55rem;\n      font-size: 0.8rem;\n      font-weight: 700;\n      text-transform: uppercase;\n      letter-spacing: 0.04em;\n      color: var(--muted);\n    }\n\n    .field {\n      display: grid;\n      gap: 0.2rem;\n      margin-block-end: 0.65rem;\n    }\n    .field label {\n      font-size: 0.82rem;\n      font-weight: 600;\n    }\n    .hint {\n      font-size: 0.75rem;\n      color: var(--muted);\n    }\n    .field input[type="number"],\n    .field textarea,\n    .panel-header select {\n      font: inherit;\n      font-size: 0.84rem;\n      padding: 0.38rem 0.55rem;\n      border-radius: 0.45rem;\n      border: 1px solid var(--border);\n      background: var(--surface);\n      color: var(--text);\n      width: 100%;\n    }\n    .field textarea {\n      font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;\n      resize: vertical;\n    }\n    .check-row {\n      display: flex;\n      align-items: center;\n      gap: 0.45rem;\n      font-size: 0.83rem;\n      margin-block: 0.4rem 0.65rem;\n    }\n    .adv-params summary {\n      font-size: 0.82rem;\n      font-weight: 600;\n      color: var(--muted);\n      cursor: pointer;\n      margin-block-end: 0.45rem;\n    }\n\n    .error-msg {\n      display: none;\n      font-size: 0.77rem;\n      color: var(--danger);\n      margin-top: 0.15rem;\n    }\n    :is(:user-invalid, .user-invalid-fallback) {\n      border-color: var(--danger) !important;\n      background-color: var(--danger-bg) !important;\n    }\n    :is(:user-invalid, .user-invalid-fallback) + .error-msg {\n      display: block;\n    }\n    :is(:user-valid, .user-valid-fallback) {\n      border-color: var(--ok);\n    }\n\n    .jobs-head {\n      display: flex;\n      align-items: center;\n      justify-content: space-between;\n      margin-bottom: 0.45rem;\n    }\n    .jobs-list {\n      display: grid;\n      grid-template-columns: minmax(0, 1fr);\n      gap: 0.45rem;\n    }\n    .job-card {\n      text-align: left;\n      padding: 0.5rem 0.6rem;\n      border-radius: 0.5rem;\n      border: 1px solid var(--border);\n      background: var(--surface-2);\n      display: grid;\n      min-width: 0;\n      gap: 0.2rem;\n      width: 100%;\n      font-weight: 400;\n    }\n    .job-card.is-active {\n      border-color: var(--brand);\n      outline: 1px solid var(--brand);\n    }\n    .job-top {\n      display: flex;\n      justify-content: space-between;\n      align-items: center;\n      min-width: 0;\n      gap: 0.4rem;\n      font-size: 0.8rem;\n      font-weight: 600;\n    }\n    .job-name {\n      min-width: 0;\n      overflow: hidden;\n      text-overflow: ellipsis;\n      white-space: nowrap;\n    }\n    .job-meta {\n      font-size: 0.74rem;\n      color: var(--muted);\n      display: flex;\n      justify-content: space-between;\n      gap: 0.35rem;\n    }\n    .job-badge {\n      font-size: 0.7rem;\n      padding: 0.05rem 0.38rem;\n      border-radius: 999px;\n      border: 1px solid var(--border);\n      flex: none;\n    }\n    .job-badge.completed, .job-badge.skipped { color: var(--ok); border-color: var(--ok); }\n    .job-badge.failed { color: var(--danger); border-color: var(--danger); }\n\n    .about-card {\n      font-size: 0.79rem;\n      color: var(--muted);\n      padding: 0.65rem;\n      border-radius: 0.5rem;\n      background: var(--surface-2);\n      border: 1px solid var(--border);\n      overflow-wrap: anywhere;\n    }\n    .about-card p { margin: 0 0 0.4rem; }\n    .about-card p:last-child { margin-bottom: 0; }\n\n    .document-panels {\n      display: grid;\n      grid-template-columns: minmax(14rem, var(--left-ratio)) 8px minmax(14rem, 1fr);\n      min-height: 0;\n      min-width: 0;\n    }\n    .file-panel, .result-panel {\n      display: flex;\n      flex-direction: column;\n      min-height: 0;\n      min-width: 0;\n      background: var(--surface);\n    }\n\n    .splitter {\n      background: var(--border);\n      cursor: col-resize;\n      position: relative;\n    }\n    .splitter:hover, .splitter:focus-visible {\n      background: var(--brand);\n      outline: none;\n    }\n\n    .file-tools {\n      display: flex;\n      align-items: center;\n      gap: 0.55rem;\n      font-weight: 500;\n      font-size: 0.8rem;\n    }\n    .file-tools select {\n      max-width: 14rem;\n      width: auto;\n      padding: 0.2rem 0.4rem;\n    }\n    .inline-check {\n      display: inline-flex;\n      align-items: center;\n      gap: 0.3rem;\n      cursor: pointer;\n    }\n\n    .file-viewport {\n      flex: 1;\n      overflow: auto;\n      padding: 0.9rem;\n      display: grid;\n      gap: 0.9rem;\n      align-content: start;\n    }\n\n    .upload-dropzone {\n      width: 100%;\n      min-height: 15rem;\n      border: 2px dashed var(--border);\n      border-radius: 0.75rem;\n      background: var(--surface-2);\n      color: var(--muted);\n      display: grid;\n      place-content: center;\n      justify-items: center;\n      gap: 0.45rem;\n      padding: 1.5rem;\n      text-align: center;\n    }\n    .upload-dropzone.is-compact {\n      min-height: auto;\n      padding: 0.7rem 1rem;\n      display: flex;\n      justify-content: space-between;\n      align-items: center;\n    }\n    .upload-dropzone:hover {\n      border-color: var(--brand);\n      color: var(--text);\n    }\n    .upload-title { font-size: 0.95rem; font-weight: 700; color: var(--text); }\n\n    .page-card {\n      margin: 0;\n      border: 1px solid var(--border);\n      border-radius: 0.65rem;\n      overflow: hidden;\n      background: var(--surface-2);\n    }\n    .page-cap {\n      padding: 0.35rem 0.65rem;\n      font-size: 0.78rem;\n      font-weight: 600;\n      color: var(--muted);\n      border-block-end: 1px solid var(--border);\n      display: flex;\n      justify-content: space-between;\n    }\n    .page-stage {\n      position: relative;\n      background: #fff;\n      line-height: 0;\n    }\n    .page-img {\n      display: block;\n      width: 100%;\n      height: auto;\n    }\n    .overlay-svg {\n      position: absolute;\n      inset: 0;\n      width: 100%;\n      height: 100%;\n      pointer-events: none;\n    }\n    .overlay-svg.is-hidden { display: none; }\n    .bbox-group { pointer-events: auto; cursor: pointer; }\n    .bbox-rect {\n      fill-opacity: 0.1;\n      stroke-width: 2.2;\n      vector-effect: non-scaling-stroke;\n    }\n    .bbox-group.is-selected .bbox-rect {\n      fill-opacity: 0.25;\n      stroke-width: 4;\n    }\n    .bbox-tag {\n      font-size: 17px;\n      font-weight: 700;\n      font-family: ui-monospace, SFMono-Regular, monospace;\n      fill: #ffffff;\n    }\n\n    .tabs {\n      display: flex;\n      gap: 0.25rem;\n      overflow-x: auto;\n      align-items: stretch;\n    }\n    [role="tab"] {\n      position: relative;\n      flex: none;\n      border: 0;\n      border-radius: 0.35rem 0.35rem 0 0;\n      background: transparent;\n      color: var(--muted);\n      padding: 0.38rem 0.75rem;\n      font-size: 0.85rem;\n      font-weight: 600;\n    }\n    [role="tab"][aria-selected="true"] {\n      color: var(--brand);\n    }\n    [role="tab"][aria-selected="true"]::after {\n      content: "";\n      position: absolute;\n      inset: auto 0 0;\n      border-block-end: 3px solid var(--brand);\n    }\n\n    .panel-stack {\n      display: grid;\n      flex: 1;\n      min-height: 0;\n    }\n    [role="tabpanel"] {\n      grid-area: 1 / 1;\n      min-height: 0;\n      overflow: auto;\n    }\n    .panel-inner {\n      padding: 1rem 1.15rem;\n    }\n\n    .empty-result {\n      padding: 3rem 1rem;\n      text-align: center;\n      color: var(--muted);\n      font-size: 0.92rem;\n    }\n\n    .md-body {\n      font-size: 0.93rem;\n      line-height: 1.68;\n      overflow-wrap: anywhere;\n    }\n    .md-body h1, .md-body h2, .md-body h3, .md-body h4 {\n      margin: 1.1em 0 0.45em;\n      line-height: 1.3;\n    }\n    .md-body h1 { font-size: 1.45rem; border-block-end: 1px solid var(--border); padding-block-end: 0.25rem; }\n    .md-body h2 { font-size: 1.22rem; }\n    .md-body h3 { font-size: 1.08rem; }\n    .md-body pre {\n      background: var(--surface-2);\n      border: 1px solid var(--border);\n      border-radius: 0.5rem;\n      padding: 0.75rem;\n      overflow-x: auto;\n      font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;\n      font-size: 0.83rem;\n    }\n    .md-body code {\n      font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;\n      background: var(--surface-2);\n      padding: 0.08rem 0.3rem;\n      border-radius: 0.25rem;\n      font-size: 0.87em;\n    }\n    .md-body pre code { background: transparent; padding: 0; }\n    .md-body blockquote {\n      margin: 0.7rem 0;\n      padding: 0.3rem 0.85rem;\n      border-inline-start: 3px solid var(--brand);\n      background: var(--surface-2);\n      color: var(--muted);\n    }\n    .table-wrap {\n      overflow-x: auto;\n      margin: 0.85rem 0;\n    }\n    .md-body table {\n      border-collapse: collapse;\n      width: 100%;\n      font-size: 0.84rem;\n    }\n    .md-body th, .md-body td {\n      border: 1px solid var(--border);\n      padding: 0.38rem 0.55rem;\n      text-align: left;\n    }\n    .md-body th { background: var(--surface-2); font-weight: 700; }\n    .md-body img {\n      max-width: 100%;\n      height: auto;\n      border-radius: 0.4rem;\n      border: 1px solid var(--border);\n      margin-block: 0.45rem;\n    }\n\n    .source-pre {\n      margin: 0;\n      white-space: pre-wrap;\n      overflow-wrap: anywhere;\n      font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;\n      font-size: 0.83rem;\n      line-height: 1.55;\n      background: var(--surface-2);\n      border: 1px solid var(--border);\n      border-radius: 0.55rem;\n      padding: 0.9rem;\n    }\n\n    .layout-list {\n      display: grid;\n      gap: 0.55rem;\n    }\n    .block-card {\n      border: 1px solid var(--border);\n      border-radius: 0.55rem;\n      padding: 0.55rem 0.7rem;\n      background: var(--surface-2);\n      cursor: pointer;\n      display: grid;\n      gap: 0.28rem;\n      text-align: left;\n      width: 100%;\n      font-weight: 400;\n    }\n    .block-card.is-selected {\n      border-color: var(--brand);\n      outline: 2px solid color-mix(in srgb, var(--brand) 50%, transparent);\n    }\n    .block-head {\n      display: flex;\n      align-items: center;\n      justify-content: space-between;\n      gap: 0.45rem;\n      font-size: 0.78rem;\n    }\n    .block-order {\n      display: inline-grid;\n      place-items: center;\n      min-width: 1.45rem;\n      height: 1.45rem;\n      padding: 0 0.3rem;\n      border-radius: 999px;\n      background: var(--brand);\n      color: var(--brand-text);\n      font-weight: 700;\n      font-size: 0.74rem;\n    }\n    .block-label { font-weight: 700; }\n    .block-coords {\n      font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;\n      color: var(--muted);\n      font-size: 0.72rem;\n    }\n    .block-snippet {\n      font-size: 0.82rem;\n      color: var(--text);\n      display: -webkit-box;\n      -webkit-line-clamp: 3;\n      -webkit-box-orient: vertical;\n      overflow: hidden;\n    }\n\n    .raw-details { margin-top: 1rem; }\n    .raw-details summary { cursor: pointer; font-weight: 600; font-size: 0.84rem; color: var(--muted); margin-bottom: 0.5rem; }\n\n    .drop-overlay {\n      position: fixed;\n      inset: 0;\n      z-index: 50;\n      display: grid;\n      place-items: center;\n      background: color-mix(in srgb, var(--bg) 82%, transparent);\n      backdrop-filter: blur(4px);\n    }\n    .drop-overlay[hidden] { display: none; }\n    .drop-box {\n      padding: 2rem 3rem;\n      border-radius: 1rem;\n      border: 2px dashed var(--brand);\n      background: var(--surface);\n      text-align: center;\n      display: grid;\n      gap: 0.35rem;\n    }\n\n    .progress-toast {\n      position: fixed;\n      right: 1.1rem;\n      bottom: 1.1rem;\n      z-index: 40;\n      min-width: 16.5rem;\n      max-width: 24rem;\n      padding: 0.75rem 0.95rem;\n      border-radius: 0.7rem;\n      background: var(--surface);\n      border: 1px solid var(--border);\n      box-shadow: 0 8px 24px rgb(0 0 0 / 0.16);\n      font-size: 0.83rem;\n    }\n    .progress-toast[hidden] { display: none; }\n    .progress-steps {\n      margin: 0;\n      padding: 0;\n      list-style: none;\n      display: grid;\n      gap: 0.38rem;\n    }\n    .progress-steps li {\n      display: flex;\n      align-items: center;\n      gap: 0.45rem;\n      color: var(--muted);\n    }\n    .progress-steps li.is-active { color: var(--text); font-weight: 700; }\n    .progress-steps li.is-done { color: var(--ok); }\n    .progress-steps li.is-fail { color: var(--danger); }\n    .dot {\n      width: 0.58rem;\n      height: 0.58rem;\n      border-radius: 999px;\n      background: currentColor;\n      flex: none;\n    }\n\n    .banner-err {\n      padding: 0.65rem 0.8rem;\n      border-radius: 0.5rem;\n      background: var(--danger-bg);\n      color: var(--danger);\n      border: 1px solid color-mix(in srgb, var(--danger) 40%, transparent);\n      font-size: 0.83rem;\n      margin-bottom: 0.85rem;\n    }\n\n    @media (max-width: 960px) {\n      .workspace {\n        grid-template-columns: 1fr;\n        grid-template-rows: auto 1fr;\n      }\n      .config-panel {\n        border-inline-end: 0;\n        border-block-end: 1px solid var(--border);\n        max-height: 40vh;\n      }\n      .document-panels {\n        grid-template-columns: 1fr;\n        grid-template-rows: minmax(14rem, 1fr) minmax(16rem, 1fr);\n      }\n      .splitter { display: none; }\n    }\n  </style>\n</head>\n<body>\n  <main class="app-shell">\n    <header class="app-header">\n      <div class="brand">\n        <h1>Unlimited-<span class="brand-accent">OCR 2</span> Online Demo</h1>\n        <span class="status-pill" id="health-pill">连接中...</span>\n      </div>\n      <div class="header-actions">\n        <button type="button" class="primary" id="pick-file-btn">上传 PDF / 图片</button>\n        <button type="button" id="copy-md-btn" disabled>复制 Markdown</button>\n        <button type="button" id="dl-md-btn" disabled>下载 Markdown</button>\n        <button type="button" id="dl-raw-btn" disabled>下载 Raw</button>\n      </div>\n    </header>\n\n    <div class="workspace">\n      <aside class="config-panel" aria-label="参数与历史任务">\n        <div class="panel-header"><span>识别配置与任务</span></div>\n        <div class="config-scroll scroller">\n          <form id="ocr-form" novalidate>\n            <h2 class="section-title">转换参数</h2>\n            <div class="field">\n              <label for="opt-dpi">DPI</label>\n              <span id="dpi-hint" class="hint">范围 72–400（默认 200）</span>\n              <input id="opt-dpi" name="dpi" type="number" min="72" max="400" step="1" value="200" required\n                     aria-describedby="dpi-hint" data-error="dpi-error">\n              <div id="dpi-error" class="error-msg"><span role="img" aria-label="错误">⚠</span> 请输入 72–400 的整数 DPI。</div>\n            </div>\n\n            <div class="field">\n              <label for="opt-chunk">每组页数 (pages_per_chunk)</label>\n              <span id="chunk-hint" class="hint">范围 1–100（默认 10，越大越占显存）</span>\n              <input id="opt-chunk" name="pages_per_chunk" type="number" min="1" max="100" step="1" value="10" required\n                     aria-describedby="chunk-hint" data-error="chunk-error">\n              <div id="chunk-error" class="error-msg"><span role="img" aria-label="错误">⚠</span> 请输入 1–100 的整数页数。</div>\n            </div>\n\n            <div class="field">\n              <label for="opt-tokens">每页 Token 上限 (tokens_per_page)</label>\n              <span id="tokens-hint" class="hint">范围 32–20000（默认 8000）</span>\n              <input id="opt-tokens" name="tokens_per_page" type="number" min="32" max="20000" step="1" value="8000" required\n                     aria-describedby="tokens-hint" data-error="tokens-error">\n              <div id="tokens-error" class="error-msg"><span role="img" aria-label="错误">⚠</span> 请输入 32–20000 的整数。</div>\n            </div>\n\n            <label class="check-row">\n              <input id="opt-overwrite" type="checkbox">\n              <span>强制重跑已有结果 (overwrite)</span>\n            </label>\n\n            <details class="adv-params">\n              <summary>高级推理参数 (inference_params JSON)</summary>\n              <div class="field">\n                <label for="opt-params">JSON 对象（留空使用默认）</label>\n                <span id="params-hint" class="hint">例: {"temperature": 0.0, "crop_mode": false}</span>\n                <textarea id="opt-params" rows="3" autocorrect="off" autocapitalize="none" spellcheck="false"\n                          placeholder="{}" aria-describedby="params-hint" data-error="params-error"></textarea>\n                <div id="params-error" class="error-msg"><span role="img" aria-label="错误">⚠</span> 请输入合法的 JSON 对象。</div>\n              </div>\n            </details>\n          </form>\n\n          <section aria-label="最近任务">\n            <div class="jobs-head">\n              <h2 class="section-title" style="margin:0">最近任务 (Drive)</h2>\n              <button type="button" id="refresh-jobs-btn">刷新</button>\n            </div>\n            <div class="jobs-list" id="jobs-list">\n              <div class="hint">正在加载任务列表...</div>\n            </div>\n          </section>\n\n          <section class="about-card" aria-label="关于模型">\n            <p><strong>模型</strong>：baidu/Unlimited-OCR-2（Transformers BF16）</p>\n            <p>上传的文件分块写入 Google Drive 并持久化保存结果、裁剪图片与版面块坐标。</p>\n          </section>\n        </div>\n      </aside>\n\n      <div class="document-panels" id="document-panels">\n        <section class="file-panel" aria-label="原文件与版面">\n          <div class="panel-header">\n            <span id="file-panel-title">原文件</span>\n            <div class="file-tools">\n              <select id="file-select" aria-label="选择任务中的文件" hidden></select>\n              <label class="inline-check">\n                <input type="checkbox" id="toggle-overlay" checked>\n                <span>版面框</span>\n              </label>\n              <span id="page-count-badge" class="hint"></span>\n            </div>\n          </div>\n          <div class="file-viewport scroller" id="file-viewport" tabindex="0">\n            <input id="file-input" type="file" hidden accept=".pdf,.png,.jpg,.jpeg,.webp,.bmp,.tif,.tiff">\n            <button type="button" class="upload-dropzone" id="upload-dropzone">\n              <span class="upload-title">拖拽 PDF 或图片到此处测试</span>\n              <span>或点击选择本地文件（支持 PDF / PNG / JPEG / WEBP / BMP / TIFF）</span>\n            </button>\n            <div id="file-pages"></div>\n          </div>\n        </section>\n\n        <div class="splitter" id="splitter" role="separator" aria-label="调整左右面板宽度"\n             aria-orientation="vertical" aria-valuemin="20" aria-valuemax="80" aria-valuenow="50" tabindex="0"></div>\n\n        <section class="result-panel" aria-label="解析结果">\n          <div class="panel-header">\n            <div class="tabs" id="result-tabs" role="tablist" focusgroup="tablist nomemory" aria-label="解析结果视图">\n              <button type="button" class="tab" role="tab" id="tab-preview" aria-selected="true" focusgroupstart data-tab="preview">预览</button>\n              <button type="button" class="tab" role="tab" id="tab-source" aria-selected="false" data-tab="source">源码</button>\n              <button type="button" class="tab" role="tab" id="tab-layout" aria-selected="false" data-tab="layout">版面</button>\n            </div>\n            <div class="file-tools">\n              <span id="result-meta" class="hint"></span>\n              <button type="button" id="resume-job-btn" hidden>断点恢复</button>\n            </div>\n          </div>\n          <div class="panel-stack">\n            <div role="tabpanel" id="panel-preview" class="scroller" aria-labelledby="tab-preview" tabindex="0">\n              <div class="panel-inner" id="preview-inner">\n                <div class="empty-result">上传文件或选择左侧已有任务，即可查看渲染后的 Markdown 与表格结果。</div>\n              </div>\n            </div>\n            <div role="tabpanel" id="panel-source" class="scroller" aria-labelledby="tab-source" hidden="until-found">\n              <div class="panel-inner" id="source-inner">\n                <div class="empty-result">暂无 Markdown 源码。</div>\n              </div>\n            </div>\n            <div role="tabpanel" id="panel-layout" class="scroller" aria-labelledby="tab-layout" hidden="until-found">\n              <div class="panel-inner" id="layout-inner">\n                <div class="empty-result">暂无版面块与阅读顺序数据。</div>\n              </div>\n            </div>\n          </div>\n        </section>\n      </div>\n    </div>\n  </main>\n\n  <div class="drop-overlay" id="drop-overlay" hidden>\n    <div class="drop-box">\n      <strong>释放文件以开始 Unlimited-OCR 2 识别</strong>\n      <span class="hint">支持 PDF / PNG / JPEG / WEBP / BMP / TIFF</span>\n    </div>\n  </div>\n\n  <div class="progress-toast" id="progress-toast" role="status" aria-live="polite" aria-atomic="true" hidden>\n    <ol class="progress-steps">\n      <li id="step-upload"><span class="dot" aria-hidden="true"></span><span id="step-upload-text">等待上传...</span></li>\n      <li id="step-parse"><span class="dot" aria-hidden="true"></span><span id="step-parse-text">等待推理...</span></li>\n    </ol>\n  </div>\n\n  <script nonce="__NONCE__">\n    (() => {\n      const qs = (id) => document.getElementById(id);\n      const state = {\n        job: null,\n        fileIndex: 0,\n        preview: null,\n        selectedBlock: null,\n        busy: false,\n      };\n\n      const LABEL_COLORS = {\n        title: "#c86300",\n        text: "#0b57d0",\n        table: "#0f766e",\n        image: "#7c3aed",\n        figure: "#7c3aed",\n        chart: "#7c3aed",\n        equation: "#be185d",\n        algorithm: "#4338ca",\n        header: "#64748b",\n        footer: "#64748b",\n        page_number: "#64748b",\n      };\n      const colorForLabel = (label) => LABEL_COLORS[label] || "#2563eb";\n\n      // --- Pure-JS SHA-256 fallback when crypto.subtle is unavailable (e.g. non-localhost HTTP) ---\n      const K256 = new Uint32Array([\n        0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5, 0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,\n        0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3, 0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,\n        0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc, 0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,\n        0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7, 0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,\n        0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13, 0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,\n        0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3, 0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,\n        0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5, 0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,\n        0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208, 0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,\n      ]);\n      function sha256FallbackHex(bytes) {\n        const len = bytes.length;\n        const bitLen = len * 8;\n        const total = ((len + 9 + 63) >> 6) << 6;\n        const buf = new Uint8Array(total);\n        buf.set(bytes);\n        buf[len] = 0x80;\n        const view = new DataView(buf.buffer);\n        view.setUint32(total - 8, Math.floor(bitLen / 0x100000000), false);\n        view.setUint32(total - 4, bitLen >>> 0, false);\n        let h0 = 0x6a09e667, h1 = 0xbb67ae85, h2 = 0x3c6ef372, h3 = 0xa54ff53a;\n        let h4 = 0x510e527f, h5 = 0x9b05688c, h6 = 0x1f83d9ab, h7 = 0x5be0cd19;\n        const w = new Uint32Array(64);\n        const rotr = (x, n) => (x >>> n) | (x << (32 - n));\n        for (let off = 0; off < total; off += 64) {\n          for (let i = 0; i < 16; i++) w[i] = view.getUint32(off + i * 4, false);\n          for (let i = 16; i < 64; i++) {\n            const s0 = rotr(w[i - 15], 7) ^ rotr(w[i - 15], 18) ^ (w[i - 15] >>> 3);\n            const s1 = rotr(w[i - 2], 17) ^ rotr(w[i - 2], 19) ^ (w[i - 2] >>> 10);\n            w[i] = (w[i - 16] + s0 + w[i - 7] + s1) >>> 0;\n          }\n          let a = h0, b = h1, c = h2, d = h3, e = h4, f = h5, g = h6, h = h7;\n          for (let i = 0; i < 64; i++) {\n            const S1 = rotr(e, 6) ^ rotr(e, 11) ^ rotr(e, 25);\n            const ch = (e & f) ^ ((~e) & g);\n            const t1 = (h + S1 + ch + K256[i] + w[i]) >>> 0;\n            const S0 = rotr(a, 2) ^ rotr(a, 13) ^ rotr(a, 22);\n            const maj = (a & b) ^ (a & c) ^ (b & c);\n            const t2 = (S0 + maj) >>> 0;\n            h = g; g = f; f = e; e = (d + t1) >>> 0;\n            d = c; c = b; b = a; a = (t1 + t2) >>> 0;\n          }\n          h0 = (h0 + a) >>> 0; h1 = (h1 + b) >>> 0; h2 = (h2 + c) >>> 0; h3 = (h3 + d) >>> 0;\n          h4 = (h4 + e) >>> 0; h5 = (h5 + f) >>> 0; h6 = (h6 + g) >>> 0; h7 = (h7 + h) >>> 0;\n        }\n        return [h0, h1, h2, h3, h4, h5, h6, h7].map((v) => v.toString(16).padStart(8, "0")).join("");\n      }\n      async function sha256Hex(arrayBuffer) {\n        if (globalThis.crypto && crypto.subtle && typeof crypto.subtle.digest === "function") {\n          const digest = await crypto.subtle.digest("SHA-256", arrayBuffer);\n          return Array.from(new Uint8Array(digest), (b) => b.toString(16).padStart(2, "0")).join("");\n        }\n        return sha256FallbackHex(new Uint8Array(arrayBuffer));\n      }\n      function bytesToBase64(bytes) {\n        let bin = "";\n        for (let i = 0; i < bytes.length; i += 0x8000) {\n          bin += String.fromCharCode(...bytes.subarray(i, i + 0x8000));\n        }\n        return btoa(bin);\n      }\n\n      // --- Form validation after interaction + ARIA error announcement ---\n      const form = qs("ocr-form");\n      const paramsTextarea = qs("opt-params");\n      function validateJsonField() {\n        const raw = paramsTextarea.value.trim();\n        if (!raw) {\n          paramsTextarea.setCustomValidity("");\n          return;\n        }\n        try {\n          const parsed = JSON.parse(raw);\n          if (!parsed || typeof parsed !== "object" || Array.isArray(parsed)) {\n            paramsTextarea.setCustomValidity("必须是 JSON 对象");\n          } else {\n            paramsTextarea.setCustomValidity("");\n          }\n        } catch {\n          paramsTextarea.setCustomValidity("JSON 语法错误");\n        }\n      }\n      paramsTextarea.addEventListener("input", validateJsonField);\n\n      const announcer = document.createElement("div");\n      announcer.setAttribute("aria-live", "polite");\n      announcer.className = "visually-hidden";\n      document.body.append(announcer);\n      let announceTimer;\n      let announcedField;\n      const flagged = new WeakSet();\n      const announced = new WeakSet();\n      const dirtyState = new WeakMap();\n      const supportsUserInvalid = CSS.supports("selector(:user-invalid)");\n\n      const linkError = (el, shown) => {\n        const errorId = el.dataset.error;\n        if (!errorId) return;\n        const ids = (el.getAttribute("aria-describedby") || "").split(/\\s+/).filter((id) => id && id !== errorId);\n        if (shown) ids.push(errorId);\n        if (ids.length) el.setAttribute("aria-describedby", ids.join(" "));\n        else el.removeAttribute("aria-describedby");\n      };\n      const announceError = (el) => {\n        const errEl = qs(el.dataset.error);\n        if (!errEl) return;\n        const label = el.labels && el.labels[0] ? el.labels[0].textContent.trim() + ": " : "";\n        clearTimeout(announceTimer);\n        announcer.textContent = "";\n        announcedField = el;\n        announceTimer = setTimeout(() => { announcer.textContent = label + errEl.textContent.trim(); }, 450);\n      };\n      const isFieldUserInvalid = (el) => {\n        if (supportsUserInvalid) return el.matches(":user-invalid");\n        const st = dirtyState.get(el);\n        return Boolean(st && st.hasBlurred && !el.validity.valid);\n      };\n      const syncAria = (el, allowAnnounce = false) => {\n        if (!el || !el.matches || !el.matches("input, select, textarea")) return;\n        if (el === paramsTextarea) validateJsonField();\n        const invalid = isFieldUserInvalid(el);\n        if (!supportsUserInvalid) {\n          const st = dirtyState.get(el);\n          el.classList.toggle("user-invalid-fallback", invalid);\n          el.classList.toggle("user-valid-fallback", Boolean(st && st.hasBlurred && el.validity.valid));\n        }\n        if (invalid) {\n          if (allowAnnounce && !announced.has(el)) {\n            announceError(el);\n            announced.add(el);\n          }\n          el.setAttribute("aria-invalid", "true");\n          linkError(el, true);\n          flagged.add(el);\n        } else if (flagged.has(el) || el.getAttribute("aria-invalid") !== "true") {\n          el.setAttribute("aria-invalid", "false");\n          linkError(el, false);\n          flagged.delete(el);\n          announced.delete(el);\n          if (announcedField === el) {\n            clearTimeout(announceTimer);\n            announcer.textContent = "";\n          }\n        }\n      };\n      form.querySelectorAll("input, textarea").forEach((el) => syncAria(el));\n      form.addEventListener("blur", (e) => {\n        if (!supportsUserInvalid && e.target.checkValidity) {\n          const st = dirtyState.get(e.target) || { hasInteracted: false, hasBlurred: false };\n          st.hasBlurred = true;\n          dirtyState.set(e.target, st);\n        }\n        syncAria(e.target, true);\n      }, true);\n      form.addEventListener("input", (e) => {\n        if (!supportsUserInvalid && e.target.checkValidity) {\n          const st = dirtyState.get(e.target) || { hasInteracted: false, hasBlurred: false };\n          st.hasInteracted = true;\n          dirtyState.set(e.target, st);\n        }\n        if (e.target.hasAttribute("aria-invalid")) syncAria(e.target);\n      });\n      form.addEventListener("invalid", (e) => {\n        if (!supportsUserInvalid) dirtyState.set(e.target, { hasInteracted: true, hasBlurred: true });\n        announced.add(e.target);\n        syncAria(e.target);\n      }, true);\n\n      // --- Accessible Tabs (focusgroup + fallback + hidden="until-found" + beforematch) ---\n      const tablist = qs("result-tabs");\n      const tabs = Array.from(tablist.querySelectorAll(\'[role="tab"]\'));\n      const panels = {\n        preview: qs("panel-preview"),\n        source: qs("panel-source"),\n        layout: qs("panel-layout"),\n      };\n      const hasFocusGroup = Object.hasOwn(HTMLElement.prototype, "focusGroup");\n      function selectTab(targetTab, { focusTab = false } = {}) {\n        const name = targetTab.dataset.tab;\n        for (const tab of tabs) {\n          const active = tab === targetTab;\n          tab.setAttribute("aria-selected", active ? "true" : "false");\n          tab.toggleAttribute("focusgroupstart", active);\n          if (!hasFocusGroup) tab.tabIndex = active ? 0 : -1;\n        }\n        for (const [key, panel] of Object.entries(panels)) {\n          const active = key === name;\n          if (active) {\n            panel.removeAttribute("hidden");\n            panel.setAttribute("tabindex", "0");\n          } else {\n            panel.setAttribute("hidden", "until-found");\n            panel.removeAttribute("tabindex");\n          }\n        }\n        targetTab.scrollIntoView({ block: "nearest", inline: "nearest", container: "nearest" });\n        if (focusTab) targetTab.focus();\n      }\n      if (!hasFocusGroup) {\n        tabs.forEach((t, idx) => { t.tabIndex = idx === 0 ? 0 : -1; });\n        tablist.addEventListener("keydown", (e) => {\n          const cur = tabs.indexOf(document.activeElement);\n          if (cur < 0) return;\n          let next = -1;\n          if (e.key === "ArrowRight") next = (cur + 1) % tabs.length;\n          else if (e.key === "ArrowLeft") next = (cur - 1 + tabs.length) % tabs.length;\n          else if (e.key === "Home") next = 0;\n          else if (e.key === "End") next = tabs.length - 1;\n          if (next >= 0) {\n            e.preventDefault();\n            selectTab(tabs[next], { focusTab: true });\n          }\n        });\n      }\n      tablist.addEventListener("click", (e) => {\n        const btn = e.target.closest(\'[role="tab"]\');\n        if (btn) selectTab(btn);\n      });\n      tablist.addEventListener("focusin", (e) => {\n        const btn = e.target.closest(\'[role="tab"]\');\n        if (btn && btn.getAttribute("aria-selected") !== "true") selectTab(btn);\n      });\n      for (const [key, panel] of Object.entries(panels)) {\n        panel.addEventListener("beforematch", () => {\n          const tab = tabs.find((t) => t.dataset.tab === key);\n          if (tab) selectTab(tab);\n        });\n      }\n\n      // --- Safe HTML table sanitization (Element.setHTML + DOMParser whitelist fallback) ---\n      const TABLE_TAGS = new Set([\n        "table", "thead", "tbody", "tfoot", "tr", "th", "td",\n        "caption", "colgroup", "col",\n        "br", "sup", "sub", "b", "strong", "i", "em", "code", "p", "span",\n      ]);\n      function cloneSafeNode(node) {\n        if (node.nodeType === Node.TEXT_NODE) return document.createTextNode(node.textContent || "");\n        if (node.nodeType !== Node.ELEMENT_NODE) return null;\n        const tag = node.tagName.toLowerCase();\n        if (!TABLE_TAGS.has(tag)) {\n          const frag = document.createDocumentFragment();\n          for (const child of node.childNodes) {\n            const c = cloneSafeNode(child);\n            if (c) frag.append(c);\n          }\n          return frag;\n        }\n        const el = document.createElement(tag);\n        if (tag === "td" || tag === "th") {\n          for (const attr of ["colspan", "rowspan"]) {\n            const val = node.getAttribute(attr);\n            if (val && /^[0-9]{1,3}$/.test(val.trim())) el.setAttribute(attr, val.trim());\n          }\n          const align = (node.getAttribute("align") || "").trim().toLowerCase();\n          if (align === "left" || align === "center" || align === "right") el.setAttribute("align", align);\n        }\n        for (const child of node.childNodes) {\n          const c = cloneSafeNode(child);\n          if (c) el.append(c);\n        }\n        return el;\n      }\n      function appendSafeTable(container, tableHtml) {\n        const wrap = document.createElement("div");\n        wrap.className = "table-wrap";\n        if ("setHTML" in Element.prototype) {\n          wrap.setHTML(tableHtml, {\n            sanitizer: {\n              elements: Array.from(TABLE_TAGS),\n              attributes: ["colspan", "rowspan", "align"],\n              replaceWithChildrenElements: ["div", "font", "center"],\n            },\n          });\n        } else {\n          const doc = new DOMParser().parseFromString(tableHtml, "text/html");\n          if (doc.body) {\n            for (const child of doc.body.childNodes) {\n              const safe = cloneSafeNode(child);\n              if (safe) wrap.append(safe);\n            }\n          }\n        }\n        container.append(wrap);\n      }\n\n      // --- Safe Markdown DOM builder (DOM nodes only) ---\n      function resolveAssetUrl(jobId, fileIndex, rel) {\n        if (!rel || /^([a-z]+:|\\/\\/|\\/)/i.test(rel)) return null;\n        const parts = rel.split("/").filter(Boolean);\n        if (!parts.length || parts.includes("..")) return null;\n        return `/api/jobs/${encodeURIComponent(jobId)}/files/${fileIndex}/asset/${parts.map(encodeURIComponent).join("/")}`;\n      }\n      function appendInlineNodes(parent, text, jobId, fileIndex) {\n        const pattern = /(!\\[([^\\]]*)\\]\\(([^)]+)\\)|`([^`]+)`|\\*\\*([^*]+)\\*\\*|\\*([^*]+)\\*)/g;\n        let last = 0;\n        for (const m of text.matchAll(pattern)) {\n          if (m.index > last) parent.append(document.createTextNode(text.slice(last, m.index)));\n          if (m[2] !== undefined) {\n            const url = resolveAssetUrl(jobId, fileIndex, m[3].trim());\n            if (url) {\n              const img = document.createElement("img");\n              img.src = url;\n              img.alt = m[2] || "识别插图";\n              img.loading = "lazy";\n              parent.append(img);\n            } else {\n              parent.append(document.createTextNode(`[图片: ${m[2] || m[3]}]`));\n            }\n          } else if (m[4] !== undefined) {\n            const code = document.createElement("code");\n            code.textContent = m[4];\n            parent.append(code);\n          } else if (m[5] !== undefined) {\n            const strong = document.createElement("strong");\n            strong.textContent = m[5];\n            parent.append(strong);\n          } else if (m[6] !== undefined) {\n            const em = document.createElement("em");\n            em.textContent = m[6];\n            parent.append(em);\n          }\n          last = m.index + m[0].length;\n        }\n        if (last < text.length) parent.append(document.createTextNode(text.slice(last)));\n      }\n      function renderMarkdownSafe(container, markdown, jobId, fileIndex) {\n        container.replaceChildren();\n        const lines = (markdown || "").replace(/\\r\\n/g, "\\n").split("\\n");\n        let i = 0;\n        while (i < lines.length) {\n          const line = lines[i];\n          if (!line.trim()) { i++; continue; }\n          if (line.trimStart().startsWith("```")) {\n            const codeLines = [];\n            i++;\n            while (i < lines.length && !lines[i].trimStart().startsWith("```")) {\n              codeLines.push(lines[i]);\n              i++;\n            }\n            if (i < lines.length) i++;\n            const pre = document.createElement("pre");\n            const code = document.createElement("code");\n            code.textContent = codeLines.join("\\n");\n            pre.append(code);\n            container.append(pre);\n            continue;\n          }\n          if (/^\\s*<table\\b/i.test(line)) {\n            const tblLines = [line];\n            if (!/<\\/table>/i.test(line)) {\n              i++;\n              while (i < lines.length) {\n                tblLines.push(lines[i]);\n                if (/<\\/table>/i.test(lines[i])) break;\n                i++;\n              }\n            }\n            i++;\n            appendSafeTable(container, tblLines.join("\\n"));\n            continue;\n          }\n          const h = line.match(/^(#{1,6})\\s+(.*)$/);\n          if (h) {\n            const tag = "h" + Math.min(6, h[1].length);\n            const el = document.createElement(tag);\n            appendInlineNodes(el, h[2], jobId, fileIndex);\n            container.append(el);\n            i++;\n            continue;\n          }\n          if (/^\\s*([-*_])\\1{2,}\\s*$/.test(line)) {\n            container.append(document.createElement("hr"));\n            i++;\n            continue;\n          }\n          if (line.startsWith(">")) {\n            const bq = document.createElement("blockquote");\n            const buf = [];\n            while (i < lines.length && lines[i].startsWith(">")) {\n              buf.push(lines[i].replace(/^>\\s?/, ""));\n              i++;\n            }\n            appendInlineNodes(bq, buf.join(" "), jobId, fileIndex);\n            container.append(bq);\n            continue;\n          }\n          const listMatch = line.match(/^\\s*([-*+]|[0-9]+\\.)\\s+(.*)$/);\n          if (listMatch) {\n            const ordered = /^[0-9]+\\./.test(listMatch[1]);\n            const listEl = document.createElement(ordered ? "ol" : "ul");\n            const itemRe = ordered ? /^\\s*[0-9]+\\.\\s+(.*)$/ : /^\\s*[-*+]\\s+(.*)$/;\n            while (i < lines.length) {\n              const m = lines[i].match(itemRe);\n              if (!m) break;\n              const li = document.createElement("li");\n              appendInlineNodes(li, m[1], jobId, fileIndex);\n              listEl.append(li);\n              i++;\n            }\n            container.append(listEl);\n            continue;\n          }\n          const para = [];\n          while (i < lines.length && lines[i].trim() && !/^(#{1,6}\\s|>|```|\\s*<table\\b|\\s*([-*+]|[0-9]+\\.)\\s)/i.test(lines[i])) {\n            para.push(lines[i]);\n            i++;\n          }\n          const p = document.createElement("p");\n          appendInlineNodes(p, para.join(" "), jobId, fileIndex);\n          container.append(p);\n        }\n      }\n\n      // --- API helpers ---\n      async function apiJson(url, opts = {}) {\n        const res = await fetch(url, opts);\n        const text = await res.text();\n        let data;\n        try { data = text ? JSON.parse(text) : {}; } catch { data = { error: text }; }\n        if (!res.ok) throw new Error(data.error || `HTTP ${res.status}`);\n        return data;\n      }\n\n      async function checkHealth() {\n        const pill = qs("health-pill");\n        try {\n          const info = await apiJson("/health");\n          pill.textContent = info.model || "已就绪";\n          pill.classList.remove("is-error");\n        } catch {\n          pill.textContent = "服务未就绪";\n          pill.classList.add("is-error");\n        }\n      }\n\n      function formatTime(ts) {\n        if (!ts) return "";\n        const d = new Date(ts * 1000);\n        return d.toLocaleTimeString([], { hour: "2-digit", minute: "2-digit" });\n      }\n\n      async function loadJobs(selectFirstIfIdle = false) {\n        const container = qs("jobs-list");\n        try {\n          const data = await apiJson("/api/jobs");\n          const jobs = data.jobs || [];\n          container.replaceChildren();\n          if (!jobs.length) {\n            const empty = document.createElement("div");\n            empty.className = "hint";\n            empty.textContent = "暂无识别记录，请先上传文件。";\n            container.append(empty);\n            return;\n          }\n          for (const j of jobs) {\n            const card = document.createElement("button");\n            card.type = "button";\n            card.className = "job-card" + (state.job && state.job.id === j.id ? " is-active" : "");\n            const top = document.createElement("div");\n            top.className = "job-top";\n            const first = (j.files && j.files[0]) || {};\n            const nameSpan = document.createElement("span");\n            nameSpan.className = "job-name";\n            nameSpan.textContent = (j.files && j.files.length > 1)\n              ? `${first.filename || j.id.slice(0, 8)} (+${j.files.length - 1})`\n              : (first.filename || j.id.slice(0, 8));\n            const badge = document.createElement("span");\n            badge.className = `job-badge ${j.status || ""}`;\n            badge.textContent = j.status || "";\n            top.append(nameSpan, badge);\n            const meta = document.createElement("div");\n            meta.className = "job-meta";\n            const idSpan = document.createElement("span");\n            idSpan.textContent = `${j.id.slice(0, 8)} · DPI ${j.options?.dpi ?? 200}`;\n            const timeSpan = document.createElement("span");\n            timeSpan.textContent = formatTime(j.created);\n            meta.append(idSpan, timeSpan);\n            card.append(top, meta);\n            card.addEventListener("click", () => openJob(j, 0));\n            container.append(card);\n          }\n          if (selectFirstIfIdle && !state.job && !state.busy && jobs[0]) {\n            await openJob(jobs[0], 0);\n          }\n        } catch (err) {\n          container.replaceChildren();\n          const msg = document.createElement("div");\n          msg.className = "hint";\n          msg.textContent = `加载任务失败: ${err.message}`;\n          container.append(msg);\n        }\n      }\n\n      function highlightBlock(blockIdx) {\n        state.selectedBlock = blockIdx;\n        document.querySelectorAll(".bbox-group").forEach((g) => {\n          g.classList.toggle("is-selected", Number(g.dataset.block) === blockIdx);\n        });\n        document.querySelectorAll(".block-card").forEach((c) => {\n          const match = Number(c.dataset.block) === blockIdx;\n          c.classList.toggle("is-selected", match);\n          if (match) c.scrollIntoView({ block: "nearest" });\n        });\n      }\n\n      function renderPages(preview) {\n        const pagesWrap = qs("file-pages");\n        const dropzone = qs("upload-dropzone");\n        pagesWrap.replaceChildren();\n        const count = Number(preview?.page_count || 0);\n        qs("page-count-badge").textContent = count ? `共 ${count} 页` : "";\n        dropzone.classList.toggle("is-compact", Boolean(preview));\n        if (!preview) return;\n        if (!preview.source_available) {\n          const note = document.createElement("div");\n          note.className = "hint";\n          note.textContent = "原始文件已不在 Drive 中，仅展示识别结果。";\n          pagesWrap.append(note);\n          return;\n        }\n        const byPage = new Map();\n        for (const blk of preview.blocks || []) {\n          if (!blk.bbox) continue;\n          if (!byPage.has(blk.page)) byPage.set(blk.page, []);\n          byPage.get(blk.page).push(blk);\n        }\n        const showBoxes = qs("toggle-overlay").checked;\n        const pageTotal = Math.max(count, 1);\n        for (let p = 1; p <= pageTotal; p++) {\n          const fig = document.createElement("figure");\n          fig.className = "page-card";\n          const cap = document.createElement("figcaption");\n          cap.className = "page-cap";\n          const leftCap = document.createElement("span");\n          leftCap.textContent = `第 ${p} / ${pageTotal} 页`;\n          const rightCap = document.createElement("span");\n          const pageBlocks = byPage.get(p) || [];\n          rightCap.textContent = pageBlocks.length ? `${pageBlocks.length} 个版面块` : "";\n          cap.append(leftCap, rightCap);\n\n          const stage = document.createElement("div");\n          stage.className = "page-stage";\n          const img = document.createElement("img");\n          img.className = "page-img";\n          img.alt = `${preview.filename || "文档"} 第 ${p} 页`;\n          img.loading = "lazy";\n          img.src = `/api/jobs/${encodeURIComponent(preview.job_id)}/files/${preview.index}/pages/${p}`;\n\n          const svg = document.createElementNS("http://www.w3.org/2000/svg", "svg");\n          svg.setAttribute("viewBox", "0 0 1000 1000");\n          svg.setAttribute("preserveAspectRatio", "none");\n          svg.setAttribute("class", "overlay-svg" + (showBoxes ? "" : " is-hidden"));\n          for (const blk of pageBlocks) {\n            const [x1, y1, x2, y2] = blk.bbox;\n            const col = colorForLabel(blk.base_label);\n            const g = document.createElementNS("http://www.w3.org/2000/svg", "g");\n            g.setAttribute("class", "bbox-group");\n            g.dataset.block = String(blk.index);\n            const rect = document.createElementNS("http://www.w3.org/2000/svg", "rect");\n            rect.setAttribute("x", String(x1));\n            rect.setAttribute("y", String(y1));\n            rect.setAttribute("width", String(Math.max(4, x2 - x1)));\n            rect.setAttribute("height", String(Math.max(4, y2 - y1)));\n            rect.setAttribute("fill", col);\n            rect.setAttribute("stroke", col);\n            rect.setAttribute("class", "bbox-rect");\n\n            const tagBg = document.createElementNS("http://www.w3.org/2000/svg", "rect");\n            const labelTxt = String(blk.index);\n            const tagW = Math.max(24, 14 + labelTxt.length * 11);\n            tagBg.setAttribute("x", String(x1));\n            tagBg.setAttribute("y", String(Math.max(0, y1 - 22)));\n            tagBg.setAttribute("width", String(tagW));\n            tagBg.setAttribute("height", "22");\n            tagBg.setAttribute("rx", "3");\n            tagBg.setAttribute("fill", col);\n\n            const txt = document.createElementNS("http://www.w3.org/2000/svg", "text");\n            txt.setAttribute("x", String(x1 + 5));\n            txt.setAttribute("y", String(Math.max(16, y1 - 6)));\n            txt.setAttribute("class", "bbox-tag");\n            txt.textContent = labelTxt;\n\n            g.append(rect, tagBg, txt);\n            g.addEventListener("click", () => {\n              highlightBlock(blk.index);\n              selectTab(qs("tab-layout"));\n            });\n            svg.append(g);\n          }\n          stage.append(img, svg);\n          fig.append(cap, stage);\n          pagesWrap.append(fig);\n        }\n      }\n\n      function renderPreviewResult(preview) {\n        const prevInner = qs("preview-inner");\n        const srcInner = qs("source-inner");\n        const layInner = qs("layout-inner");\n        prevInner.replaceChildren();\n        srcInner.replaceChildren();\n        layInner.replaceChildren();\n\n        const hasMd = Boolean(preview && typeof preview.markdown === "string" && preview.markdown.length > 0);\n        const hasRaw = Boolean(preview && typeof preview.raw === "string" && preview.raw.length > 0);\n        qs("copy-md-btn").disabled = !hasMd;\n        qs("dl-md-btn").disabled = !hasMd;\n        qs("dl-raw-btn").disabled = !hasRaw;\n\n        const metaParts = [];\n        if (preview?.status) metaParts.push(preview.status);\n        if (preview?.pages) metaParts.push(`${preview.pages} 页`);\n        if (preview?.tokens) metaParts.push(`${preview.tokens} tokens`);\n        qs("result-meta").textContent = metaParts.join(" · ");\n\n        const canResume = preview && (preview.status === "failed" || preview.status === "interrupted");\n        qs("resume-job-btn").hidden = !canResume;\n\n        if (!preview) return;\n        if (preview.error) {\n          const errDiv = document.createElement("div");\n          errDiv.className = "banner-err";\n          errDiv.textContent = `错误：${preview.error}`;\n          prevInner.append(errDiv);\n        }\n\n        if (hasMd) {\n          const pre = document.createElement("pre");\n          pre.className = "source-pre";\n          pre.textContent = preview.markdown;\n          srcInner.append(pre);\n        } else {\n          const emptyS = document.createElement("div");\n          emptyS.className = "empty-result";\n          emptyS.textContent = "暂无 Markdown 源码。";\n          srcInner.append(emptyS);\n        }\n\n        const blocks = preview.blocks || [];\n        if (blocks.length) {\n          const list = document.createElement("div");\n          list.className = "layout-list";\n          for (const blk of blocks) {\n            const card = document.createElement("button");\n            card.type = "button";\n            card.className = "block-card";\n            card.dataset.block = String(blk.index);\n            const head = document.createElement("div");\n            head.className = "block-head";\n            const left = document.createElement("span");\n            left.style.display = "inline-flex";\n            left.style.alignItems = "center";\n            left.style.gap = "0.4rem";\n            const ord = document.createElement("span");\n            ord.className = "block-order";\n            ord.style.background = colorForLabel(blk.base_label);\n            ord.textContent = String(blk.index);\n            const lbl = document.createElement("span");\n            lbl.className = "block-label";\n            lbl.textContent = blk.level ? `${blk.label} (H${blk.level}) · P${blk.page}` : `${blk.label} · P${blk.page}`;\n            left.append(ord, lbl);\n            const coords = document.createElement("span");\n            coords.className = "block-coords";\n            coords.textContent = blk.bbox ? `[${blk.bbox.join(", ")}]` : "无坐标";\n            head.append(left, coords);\n            const snippet = document.createElement("div");\n            snippet.className = "block-snippet";\n            snippet.textContent = blk.content || "(图像/空块)";\n            card.append(head, snippet);\n            card.addEventListener("click", () => {\n              highlightBlock(blk.index);\n              const targetBox = document.querySelector(`.bbox-group[data-block="${blk.index}"]`);\n              if (targetBox) targetBox.scrollIntoView({ block: "center", behavior: "smooth" });\n            });\n            list.append(card);\n          }\n          layInner.append(list);\n        } else {\n          const emptyL = document.createElement("div");\n          emptyL.className = "empty-result";\n          emptyL.textContent = "暂无版面块数据。";\n          layInner.append(emptyL);\n        }\n\n        if (hasRaw) {\n          const det = document.createElement("details");\n          det.className = "raw-details";\n          const sum = document.createElement("summary");\n          sum.textContent = "查看模型原始输出 (result.raw.txt)";\n          const pre = document.createElement("pre");\n          pre.className = "source-pre";\n          pre.textContent = preview.raw;\n          det.append(sum, pre);\n          layInner.append(det);\n        }\n\n        if (hasMd) {\n          const body = document.createElement("div");\n          body.className = "md-body";\n          try {\n            renderMarkdownSafe(body, preview.markdown, preview.job_id, preview.index);\n            prevInner.append(body);\n          } catch (err) {\n            const errDiv = document.createElement("div");\n            errDiv.className = "banner-err";\n            errDiv.textContent = `预览渲染异常，已切换纯文本展示：${err && err.message ? err.message : err}`;\n            const fallbackPre = document.createElement("pre");\n            fallbackPre.className = "source-pre";\n            fallbackPre.textContent = preview.markdown;\n            prevInner.append(errDiv, fallbackPre);\n          }\n        } else {\n          const emptyP = document.createElement("div");\n          emptyP.className = "empty-result";\n          emptyP.textContent = (preview.status === "queued" || preview.status === "running")\n            ? `任务正在处理中 (${preview.pages_done ?? 0}/${preview.pages_total ?? "?"} 页)...`\n            : "暂无 Markdown 输出。";\n          prevInner.append(emptyP);\n        }\n      }\n\n      async function loadFilePreview(jobId, fileIndex) {\n        const preview = await apiJson(`/api/jobs/${encodeURIComponent(jobId)}/files/${fileIndex}/preview`);\n        state.fileIndex = fileIndex;\n        state.preview = preview;\n        qs("file-panel-title").textContent = preview.filename ? `原文件 · ${preview.filename}` : "原文件";\n        renderPages(preview);\n        renderPreviewResult(preview);\n      }\n\n      async function openJob(job, fileIndex = 0) {\n        state.job = job;\n        const sel = qs("file-select");\n        const files = job.files || [];\n        sel.replaceChildren();\n        files.forEach((f, idx) => {\n          const opt = document.createElement("option");\n          opt.value = String(idx);\n          opt.textContent = f.filename || f.relative_output || `文件 ${idx + 1}`;\n          sel.append(opt);\n        });\n        sel.hidden = files.length <= 1;\n        sel.value = String(fileIndex);\n        await loadFilePreview(job.id, fileIndex);\n        await loadJobs(false);\n      }\n\n      // --- Upload & Poll flow ---\n      function setProgress({ visible = true, uploadText = "", uploadState = "", parseText = "", parseState = "" }) {\n        const toast = qs("progress-toast");\n        toast.hidden = !visible;\n        if (!visible) return;\n        const upLi = qs("step-upload");\n        const paLi = qs("step-parse");\n        upLi.className = uploadState;\n        paLi.className = parseState;\n        if (uploadText) qs("step-upload-text").textContent = uploadText;\n        if (parseText) qs("step-parse-text").textContent = parseText;\n      }\n\n      function readFormOptions() {\n        validateJsonField();\n        const valid = form.checkValidity();\n        if (!valid) {\n          form.reportValidity();\n          return null;\n        }\n        const rawParams = paramsTextarea.value.trim();\n        return {\n          dpi: Number(qs("opt-dpi").value),\n          pages_per_chunk: Number(qs("opt-chunk").value),\n          tokens_per_page: Number(qs("opt-tokens").value),\n          overwrite: qs("opt-overwrite").checked,\n          inference_params: rawParams ? JSON.parse(rawParams) : {},\n        };\n      }\n\n      async function uploadChunkedFile(file) {\n        const buf = await file.arrayBuffer();\n        const bytes = new Uint8Array(buf);\n        const digest = await sha256Hex(buf);\n        const CHUNK = 384 * 1024;\n        let offset = 0;\n        if (bytes.length === 0) throw new Error("空文件无法识别");\n        while (offset < bytes.length) {\n          const slice = bytes.subarray(offset, offset + CHUNK);\n          const nextOffset = offset + slice.length;\n          const final = nextOffset >= bytes.length;\n          const pct = Math.round((nextOffset / bytes.length) * 100);\n          setProgress({\n            visible: true,\n            uploadText: `正在分块上传 (${pct}%)...`,\n            uploadState: "is-active",\n            parseText: "等待推理...",\n            parseState: "",\n          });\n          const res = await apiJson("/api/upload", {\n            method: "POST",\n            headers: { "Content-Type": "application/json" },\n            body: JSON.stringify({\n              sha256: digest,\n              name: file.name,\n              offset,\n              data: bytesToBase64(slice),\n              final,\n            }),\n          });\n          if (res.complete) return res.path;\n          offset = nextOffset;\n        }\n        throw new Error("上传未完成");\n      }\n\n      async function pollJobUntilDone(jobId) {\n        while (true) {\n          const job = await apiJson(`/api/jobs/${encodeURIComponent(jobId)}`);\n          const first = (job.files && job.files[0]) || {};\n          const pgDone = first.pages_done ?? 0;\n          const pgTot = first.pages_total ?? "?";\n          if (job.status === "queued" || job.status === "running") {\n            setProgress({\n              visible: true,\n              uploadText: "上传完成",\n              uploadState: "is-done",\n              parseText: job.status === "queued" ? "排队等待 GPU..." : `正在识别 (${pgDone}/${pgTot} 页)...`,\n              parseState: "is-active",\n            });\n            await new Promise((r) => setTimeout(r, 1500));\n            continue;\n          }\n          await openJob(job, 0);\n          const failed = job.status === "failed";\n          setProgress({\n            visible: true,\n            uploadText: "上传完成",\n            uploadState: "is-done",\n            parseText: failed ? `任务失败: ${first.error || job.error || ""}` : "识别完成",\n            parseState: failed ? "is-fail" : "is-done",\n          });\n          setTimeout(() => setProgress({ visible: false }), failed ? 5000 : 2500);\n          return;\n        }\n      }\n\n      async function handleFiles(fileList) {\n        if (!fileList || !fileList.length || state.busy) return;\n        const opts = readFormOptions();\n        if (!opts) return;\n        const file = fileList[0];\n        state.busy = true;\n        qs("pick-file-btn").disabled = true;\n        let uploaded = false;\n        try {\n          const remotePath = await uploadChunkedFile(file);\n          uploaded = true;\n          setProgress({\n            visible: true,\n            uploadText: "上传完成",\n            uploadState: "is-done",\n            parseText: "已提交 GPU 队列...",\n            parseState: "is-active",\n          });\n          const job = await apiJson("/api/jobs", {\n            method: "POST",\n            headers: { "Content-Type": "application/json" },\n            body: JSON.stringify({\n              inputs: [remotePath],\n              relative_outputs: { [remotePath]: `demo/${file.name}` },\n              dpi: opts.dpi,\n              pages_per_chunk: opts.pages_per_chunk,\n              tokens_per_page: opts.tokens_per_page,\n              overwrite: opts.overwrite,\n              inference_params: opts.inference_params,\n            }),\n          });\n          await openJob(job, 0);\n          await pollJobUntilDone(job.id);\n        } catch (err) {\n          setProgress({\n            visible: true,\n            uploadText: uploaded ? "上传完成" : "上传中断",\n            uploadState: uploaded ? "is-done" : "is-fail",\n            parseText: uploaded ? `预览或提交失败: ${err.message}` : err.message,\n            parseState: "is-fail",\n          });\n          setTimeout(() => setProgress({ visible: false }), 8000);\n        } finally {\n          state.busy = false;\n          qs("pick-file-btn").disabled = false;\n        }\n      }\n\n      // --- UI Event bindings ---\n      const fileInput = qs("file-input");\n      qs("pick-file-btn").addEventListener("click", () => fileInput.click());\n      qs("upload-dropzone").addEventListener("click", () => fileInput.click());\n      fileInput.addEventListener("change", () => {\n        if (fileInput.files && fileInput.files.length) {\n          const files = fileInput.files;\n          handleFiles(files).finally(() => { fileInput.value = ""; });\n        }\n      });\n\n      let dragDepth = 0;\n      const overlay = qs("drop-overlay");\n      window.addEventListener("dragenter", (e) => {\n        if (e.dataTransfer && Array.from(e.dataTransfer.types || []).includes("Files")) {\n          e.preventDefault();\n          dragDepth++;\n          overlay.hidden = false;\n        }\n      });\n      window.addEventListener("dragover", (e) => {\n        if (e.dataTransfer && Array.from(e.dataTransfer.types || []).includes("Files")) e.preventDefault();\n      });\n      window.addEventListener("dragleave", () => {\n        dragDepth = Math.max(0, dragDepth - 1);\n        if (dragDepth === 0) overlay.hidden = true;\n      });\n      window.addEventListener("drop", (e) => {\n        e.preventDefault();\n        dragDepth = 0;\n        overlay.hidden = true;\n        if (e.dataTransfer && e.dataTransfer.files && e.dataTransfer.files.length) {\n          handleFiles(e.dataTransfer.files);\n        }\n      });\n\n      qs("file-select").addEventListener("change", (e) => {\n        if (state.job) loadFilePreview(state.job.id, Number(e.target.value));\n      });\n      qs("toggle-overlay").addEventListener("change", (e) => {\n        document.querySelectorAll(".overlay-svg").forEach((svg) => {\n          svg.classList.toggle("is-hidden", !e.target.checked);\n        });\n      });\n      qs("refresh-jobs-btn").addEventListener("click", () => loadJobs(false));\n      qs("resume-job-btn").addEventListener("click", async () => {\n        if (!state.job || state.busy) return;\n        state.busy = true;\n        try {\n          await apiJson(`/api/jobs/${encodeURIComponent(state.job.id)}/resume`, {\n            method: "POST",\n            headers: { "Content-Type": "application/json" },\n            body: "{}",\n          });\n          await pollJobUntilDone(state.job.id);\n        } catch (err) {\n          setProgress({ visible: true, uploadText: "恢复任务", uploadState: "is-done", parseText: err.message, parseState: "is-fail" });\n        } finally {\n          state.busy = false;\n        }\n      });\n\n      qs("copy-md-btn").addEventListener("click", async () => {\n        if (!state.preview?.markdown) return;\n        await navigator.clipboard.writeText(state.preview.markdown);\n        const btn = qs("copy-md-btn");\n        const old = btn.textContent;\n        btn.textContent = "已复制";\n        setTimeout(() => { btn.textContent = old; }, 1400);\n      });\n      function triggerAssetDownload(rel, fallbackName, content) {\n        if (!state.preview) return;\n        const url = resolveAssetUrl(state.preview.job_id, state.preview.index, rel);\n        const a = document.createElement("a");\n        if (url && (state.preview.assets || []).includes(rel)) {\n          a.href = url + "?download=1";\n          a.download = fallbackName;\n          a.click();\n          return;\n        }\n        const blob = new Blob([content || ""], { type: "text/plain;charset=utf-8" });\n        a.href = URL.createObjectURL(blob);\n        a.download = fallbackName;\n        a.click();\n        setTimeout(() => URL.revokeObjectURL(a.href), 1000);\n      }\n      qs("dl-md-btn").addEventListener("click", () => {\n        const stem = (state.preview?.filename || "result").replace(/\\.[^.]+$/, "");\n        triggerAssetDownload("result.md", `${stem}.md`, state.preview?.markdown || "");\n      });\n      qs("dl-raw-btn").addEventListener("click", () => {\n        const stem = (state.preview?.filename || "result").replace(/\\.[^.]+$/, "");\n        triggerAssetDownload("result.raw.txt", `${stem}.raw.txt`, state.preview?.raw || "");\n      });\n\n      // --- Resizable vertical splitter (pointer + keyboard) ---\n      const splitter = qs("splitter");\n      const docPanels = qs("document-panels");\n      function setRatio(pct) {\n        const clamped = Math.max(20, Math.min(80, pct));\n        document.documentElement.style.setProperty("--left-ratio", clamped.toFixed(1) + "%");\n        splitter.setAttribute("aria-valuenow", String(Math.round(clamped)));\n      }\n      splitter.addEventListener("pointerdown", (e) => {\n        splitter.setPointerCapture(e.pointerId);\n        const rect = docPanels.getBoundingClientRect();\n        const move = (ev) => setRatio(((ev.clientX - rect.left) / rect.width) * 100);\n        const up = () => {\n          splitter.removeEventListener("pointermove", move);\n          splitter.removeEventListener("pointerup", up);\n        };\n        splitter.addEventListener("pointermove", move);\n        splitter.addEventListener("pointerup", up);\n      });\n      splitter.addEventListener("keydown", (e) => {\n        const cur = Number(splitter.getAttribute("aria-valuenow") || 50);\n        if (e.key === "ArrowLeft") setRatio(cur - 4);\n        else if (e.key === "ArrowRight") setRatio(cur + 4);\n      });\n\n      checkHealth();\n      loadJobs(true);\n    })();\n  </script>\n</body>\n</html>\n"""\n\n\ndef render_demo_html(nonce: str) -> bytes:\n    """Render the interactive Unlimited-OCR 2 demo page with a per-request CSP nonce."""\n    return HTML_TEMPLATE.replace(\'__NONCE__\', nonce).encode(\'utf-8\')\n', encoding='utf-8')
(OCR_WORK / 'unlimited_ocr_app.py').write_text('"""Drive-native Unlimited-OCR 2 job server (Caddy protects the loopback API)."""\nfrom __future__ import annotations\n\nimport hashlib\nimport base64\nimport io\nimport json\nimport os\nimport queue\nimport re\nimport secrets\nimport sys\nimport tempfile\nimport threading\nimport time\nimport urllib.parse\nimport uuid\nimport shutil\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom pathlib import Path\nif str(Path(__file__).resolve().parent) not in sys.path:\n    sys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom ocr_parameters import validate_parameters, inference_kwargs\nfrom unlimited_ocr_demo import make_csp, parse_raw_blocks, render_demo_html\n\nEXTENSIONS = {\'.pdf\', \'.png\', \'.jpg\', \'.jpeg\', \'.webp\', \'.bmp\', \'.tif\', \'.tiff\'}\nDRIVE = Path(os.environ.get(\'CT_DRIVE_ROOT\', \'/content/drive/MyDrive\')).resolve()\nDATA = Path(os.environ.get(\'CT_DATA_DIR\', str(DRIVE / \'colab-llm/unlimited-ocr\')))\nMODEL_REVISION = \'0480966938291445591f69f08a13b041c073a96a\'\n\n\ndef drive_path(spec: str) -> Path:\n    if not isinstance(spec, str) or not spec.strip():\n        raise ValueError(\'Drive 路径不能为空\')\n    spec = spec.removeprefix(\'drive://\')\n    if spec == \'MyDrive\':\n        spec = \'.\'\n    elif spec.startswith(\'MyDrive/\'):\n        spec = spec[len(\'MyDrive/\'):]\n    path = Path(spec)\n    path = (path if path.is_absolute() else DRIVE / path).resolve()\n    if not path.is_relative_to(DRIVE):\n        raise ValueError(\'路径必须位于 /content/drive/MyDrive 内\')\n    return path\n\n\ndef atomic_json(path: Path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.\' + uuid.uuid4().hex + \'.tmp\')\n    try:\n        temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n        os.replace(temporary, path)\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef discover(specs, recursive=True, output=None):\n    if not isinstance(specs, list) or not specs:\n        raise ValueError(\'至少指定一个 Drive 文件或文件夹\')\n    found = set()\n    for spec in specs:\n        root = drive_path(spec)\n        if not root.exists():\n            raise ValueError(f\'输入不存在: {spec}\')\n        candidates = (root.rglob(\'*\') if recursive else root.iterdir()) if root.is_dir() else [root]\n        for candidate in candidates:\n            path = candidate.resolve()\n            if not path.is_relative_to(DRIVE):\n                continue\n            if output and path.is_relative_to(output):\n                continue\n            if path.is_file() and path.suffix.lower() in EXTENSIONS:\n                found.add(path)\n    if not found:\n        raise ValueError(\'未找到 PDF 或受支持的图片\')\n    return sorted(found)\n\n\ndef options(body):\n    dpi = int(body.get(\'dpi\', 200))\n    pages = int(body.get(\'pages_per_chunk\', 10))\n    tokens = int(body.get(\'tokens_per_page\', 8000))\n    if not 72 <= dpi <= 400 or not 1 <= pages <= 100 or not 32 <= tokens <= 20000:\n        raise ValueError(\'dpi: 72–400; pages_per_chunk: 1–100; tokens_per_page: 32–20000\')\n    return dict(dpi=dpi, pages_per_chunk=pages, tokens_per_page=tokens,\n                inference_params=validate_parameters(body.get(\'inference_params\', {})))\n\n\ndef file_signature(source, opts):\n    stat = source.stat()\n    return hashlib.sha256(json.dumps([str(source), stat.st_size, stat.st_mtime_ns,\n                                     opts, MODEL_REVISION]).encode()).hexdigest()\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with path.open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\nclass Checkpoints:\n    """Commit a page group only after raw text, Markdown and all assets are durable."""\n    def __init__(self, destination, signature):\n        self.destination = Path(destination)\n        self.root = self.destination / \'.checkpoints\'\n        self.signature = signature\n        try:\n            current = json.loads((self.root / \'state.json\').read_text())\n        except (OSError, ValueError):\n            current = {}\n        if current.get(\'signature\') != signature:\n            shutil.rmtree(self.root, ignore_errors=True)\n            # These names are exclusively generated by this service.\n            for path in self.destination.iterdir():\n                if re.fullmatch(r\'(chunk|native)-[0-9]+\', path.name):\n                    if path.is_symlink():\n                        path.unlink()\n                    elif path.is_dir():\n                        shutil.rmtree(path)\n            atomic_json(self.root / \'state.json\', dict(signature=signature, version=1))\n\n    def load(self, start, end):\n        try:\n            record = json.loads((self.root / f\'{start:06d}.json\').read_text())\n            if (record[\'signature\'] != self.signature or record[\'end\'] != end\n                    or not isinstance(record[\'raw\'], str) or not isinstance(record[\'markdown\'], str)\n                    or type(record[\'tokens\']) is not int or record[\'tokens\'] < 0):\n                return None\n            for name, digest in record[\'artifacts\'].items():\n                path = (self.destination / name).resolve()\n                if not path.is_relative_to(self.destination.resolve()) or sha256_file(path) != digest:\n                    return None\n            return record\n        except (OSError, ValueError, KeyError, TypeError, AttributeError):\n            return None\n\n    def save(self, start, end, raw, markdown, tokens, artifacts):\n        record = dict(signature=self.signature, end=end, raw=raw, markdown=markdown, tokens=tokens,\n                      artifacts={str(p.relative_to(self.destination)): sha256_file(p) for p in artifacts})\n        atomic_json(self.root / f\'{start:06d}.json\', record)\n        return record\n\n\nclass Engine:\n    def __init__(self):\n        import torch\n        from transformers import AutoModel, AutoTokenizer\n        self.torch = torch\n        model_dir = os.environ[\'CT_OCR_MODEL_DIR\']\n        self.tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)\n        self.model = AutoModel.from_pretrained(\n            model_dir, trust_remote_code=True, use_safetensors=True,\n            torch_dtype=torch.bfloat16).eval().cuda()\n        # Compile and exercise the actual GPU path before health can report ready.\n        from PIL import Image, ImageDraw\n        image = Image.new(\'RGB\', (512, 512), \'white\')\n        ImageDraw.Draw(image).text((32, 32), \'OCR ready\', fill=\'black\')\n        with torch.inference_mode():\n            self.model.infer(self.tokenizer, image_file=image, eval_mode=True,\n                             max_new_tokens=32, crop_mode=False)\n\n    def infer_pages(self, pages, native_output, opts):\n        # infer_multi forces return_dict=True and rewrites prompts; infer preserves all options.\n        params = inference_kwargs(opts.get(\'inference_params\', {}), len(pages), opts[\'tokens_per_page\'])\n        with self.torch.inference_mode():\n            result = self.model.infer(self.tokenizer, image_file=[str(p) for p in pages],\n                                      output_path=str(native_output), **params)\n        if isinstance(result, dict):\n            return result[\'text\'], len(result[\'token_ids\'])\n        return result, len(self.tokenizer.encode(result, add_special_tokens=False))\n\n    def convert(self, source: Path, destination: Path, opts, progress):\n        import fitz\n        from PIL import Image\n        from process_unlimitedOCR2 import convert\n        destination.mkdir(parents=True, exist_ok=True)\n        raw_chunks, markdown_chunks = [], []\n        total_tokens = 0\n        resumed_chunks = 0\n        # Overwrite uses a per-job generation: restart resumes that overwrite, a new job starts fresh.\n        checkpoint_signature = hashlib.sha256(json.dumps([\n            file_signature(source, {k: v for k, v in opts.items() if not k.startswith(\'_\')}),\n            opts.get(\'_checkpoint_generation\', \'shared\'), 1]).encode()).hexdigest()\n        checkpoints = Checkpoints(destination, checkpoint_signature)\n        with tempfile.TemporaryDirectory(prefix=\'ocr-\', dir=\'/content/unlimited-ocr\') as tmp:\n            # Render only the current page window, keeping RAM and local disk bounded.\n            pdf = fitz.open(source) if source.suffix.lower() == \'.pdf\' else None\n            raster = Image.open(source) if pdf is None else None\n            try:\n                page_count = len(pdf) if pdf is not None else getattr(raster, \'n_frames\', 1)\n                for start in range(0, page_count, opts[\'pages_per_chunk\']):\n                    end = min(start + opts[\'pages_per_chunk\'], page_count)\n                    saved = checkpoints.load(start, end)\n                    if saved is not None:\n                        raw_chunks.append(saved[\'raw\'])\n                        markdown_chunks.append(saved[\'markdown\'])\n                        total_tokens += saved[\'tokens\']\n                        resumed_chunks += 1\n                        progress(end, page_count)\n                        continue\n                    # A group without a valid commit may contain interrupted/stale assets.\n                    for prefix in (\'chunk\', \'native\'):\n                        shutil.rmtree(destination / f\'{prefix}-{start + 1:04d}\', ignore_errors=True)\n                    pages = []\n                    for index in range(start, min(start + opts[\'pages_per_chunk\'], page_count)):\n                        path = Path(tmp) / f\'page{index + 1}.png\'\n                        if pdf is not None:\n                            pdf[index].get_pixmap(dpi=opts[\'dpi\'], alpha=False).save(path)\n                        else:\n                            raster.seek(index)\n                            raster.convert(\'RGB\').save(path)\n                        pages.append(path)\n                    raw, count = self.infer_pages(pages, destination / f\'native-{start + 1:04d}\', opts)\n                    total_tokens += count\n                    # Each chunk has its own asset directory; upstream Markdown page indices restart at 1.\n                    assets = destination / f\'chunk-{start + 1:04d}\'\n                    assets.mkdir(exist_ok=True)\n                    markdown = convert(raw)\n                    for name, page, coords in re.findall(\n                            r\'(page(\\d+)_((?:\\d+_){3}\\d+)\\.jpg)\', markdown):\n                        idx = int(page) - 1\n                        if 0 <= idx < len(pages):\n                            with Image.open(pages[idx]) as image:\n                                x1, y1, x2, y2 = map(int, coords.split(\'_\'))\n                                if x2 > x1 and y2 > y1:\n                                    image.crop((x1 * image.width / 1000, y1 * image.height / 1000,\n                                                x2 * image.width / 1000, y2 * image.height / 1000)).convert(\'RGB\').save(assets / name)\n                    markdown = re.sub(r\'!\\[\\]\\((page\\d+_[\\d_]+\\.jpg)\\)\',\n                                      lambda m: f\'![]({assets.name}/{m[1]})\', markdown)\n                    artifacts = [p for prefix in (\'chunk\', \'native\')\n                                 for p in (destination / f\'{prefix}-{start + 1:04d}\').rglob(\'*\') if p.is_file()]\n                    checkpoints.save(start, end, raw, markdown, count, artifacts)\n                    raw_chunks.append(raw)\n                    markdown_chunks.append(markdown)\n                    for path in pages:\n                        path.unlink()\n                    progress(min(start + len(pages), page_count), page_count)\n                if not page_count:\n                    raise ValueError(\'文档没有页面\')\n            finally:\n                if pdf is not None:\n                    pdf.close()\n                if raster is not None:\n                    raster.close()\n        (destination / \'result.raw.txt\').write_text(\'\\n\'.join(raw_chunks), encoding=\'utf-8\')\n        (destination / \'result.md\').write_text(\'\\n\\n\'.join(markdown_chunks), encoding=\'utf-8\')\n        return dict(pages=page_count, tokens=total_tokens, chunks=len(raw_chunks), resumed_chunks=resumed_chunks)\n\n\nclass Jobs:\n    def __init__(self, engine, data=DATA, start_worker=True):\n        self.engine = engine\n        self.root = Path(data) / \'jobs\'\n        self.root.mkdir(parents=True, exist_ok=True)\n        self.lock = threading.Lock()\n        self.queue = queue.Queue()\n        recovered = []\n        for path in self.root.glob(\'*.json\'):\n            try:\n                record = json.loads(path.read_text())\n                if record[\'status\'] in (\'queued\', \'running\'):\n                    self.prepare_resume(record, automatic=True)\n                    self.save(record)\n                    recovered.append(record)\n            except (OSError, ValueError, KeyError, TypeError) as error:\n                print(f\'[recovery] skipped invalid job record {path.name}: {error}\', flush=True)\n        for record in sorted(recovered, key=lambda item: item.get(\'created\', 0)):\n            self.queue.put(record)\n        if start_worker:\n            threading.Thread(target=self.worker, daemon=True).start()\n\n    @staticmethod\n    def prepare_resume(record, automatic=False):\n        # Validate persisted settings before putting them back on the GPU queue.\n        record[\'options\'] = options(record[\'options\'])\n        if not record.get(\'files\') or not re.fullmatch(r\'[0-9a-f]{32}\', record[\'id\']):\n            raise ValueError(\'非法任务记录\')\n        record.update(status=\'queued\', resume_count=record.get(\'resume_count\', 0) + 1,\n                      resumed_at=time.time(), automatic_resume=automatic)\n        record.pop(\'error\', None)\n        record.pop(\'finished\', None)\n        for item in record[\'files\']:\n            if item[\'status\'] not in (\'completed\', \'skipped\'):\n                item[\'status\'] = \'queued\'\n                item.pop(\'error\', None)\n\n    def resume(self, job_id):\n        # One lock covers validation and persisted transition, so duplicate resume cannot enqueue twice.\n        if not re.fullmatch(r\'[0-9a-f]{32}\', job_id):\n            raise ValueError(\'非法 job ID\')\n        with self.lock:\n            path = self.root / (job_id + \'.json\')\n            record = json.loads(path.read_text())\n            if record[\'status\'] in (\'queued\', \'running\', \'completed\'):\n                raise ValueError(\'任务已在队列中、运行中或已完成\')\n            self.prepare_resume(record)\n            atomic_json(path, record)\n            self.queue.put(record)\n        return record\n\n    def get(self, job_id):\n        if not re.fullmatch(r\'[0-9a-f]{32}\', job_id):\n            raise ValueError(\'非法 job ID\')\n        with self.lock:\n            return json.loads((self.root / (job_id + \'.json\')).read_text())\n\n    def list_jobs(self, limit: int = 30) -> list[dict]:\n        items = []\n        with self.lock:\n            for path in self.root.glob(\'*.json\'):\n                try:\n                    record = json.loads(path.read_text(encoding=\'utf-8\'))\n                    if not isinstance(record, dict) or not re.fullmatch(r\'[0-9a-f]{32}\', str(record.get(\'id\') or \'\')):\n                        continue\n                except (OSError, ValueError, TypeError):\n                    continue\n                files = []\n                for idx, item in enumerate(record.get(\'files\') or ()):\n                    src = str(item.get(\'source\') or \'\')\n                    files.append({\n                        \'index\': idx,\n                        \'filename\': Path(src).name if src else (item.get(\'relative_output\') or f\'file-{idx + 1}\'),\n                        \'relative_output\': item.get(\'relative_output\'),\n                        \'status\': item.get(\'status\'),\n                        \'pages_done\': item.get(\'pages_done\'),\n                        \'pages_total\': item.get(\'pages_total\'),\n                        \'pages\': item.get(\'pages\'),\n                        \'tokens\': item.get(\'tokens\'),\n                        \'chunks\': item.get(\'chunks\'),\n                        \'error\': item.get(\'error\'),\n                    })\n                items.append({\n                    \'id\': record[\'id\'],\n                    \'status\': record.get(\'status\'),\n                    \'created\': record.get(\'created\'),\n                    \'finished\': record.get(\'finished\'),\n                    \'options\': record.get(\'options\') or {},\n                    \'overwrite\': bool(record.get(\'overwrite\', False)),\n                    \'error\': record.get(\'error\'),\n                    \'files\': files,\n                })\n        items.sort(key=lambda rec: float(rec.get(\'created\') or 0), reverse=True)\n        return items[:max(1, int(limit))]\n\n    def save(self, record):\n        with self.lock:\n            atomic_json(self.root / (record[\'id\'] + \'.json\'), record)\n\n    def submit(self, body):\n        opts = options(body)\n        output = drive_path(body.get(\'output_dir\') or str(DATA / \'outputs\'))\n        paths = discover(body.get(\'inputs\'), body.get(\'recursive\', True), output)\n        if any(output == path or output.is_relative_to(path) for path in paths):\n            raise ValueError(\'输出目录不能位于输入文件内部\')\n        job = dict(id=uuid.uuid4().hex, status=\'queued\', created=time.time(), options=opts,\n                   output_dir=str(output), overwrite=bool(body.get(\'overwrite\', False)),\n                   files=[dict(source=str(p), status=\'queued\',\n                               relative_output=body.get(\'relative_outputs\', {}).get(str(p), str(p.relative_to(DRIVE)))) for p in paths])\n        for item in job[\'files\']:\n            relative = Path(item[\'relative_output\'])\n            if relative.is_absolute() or \'..\' in relative.parts or not relative.parts:\n                raise ValueError(\'非法输出相对路径\')\n        if len({item[\'relative_output\'] for item in job[\'files\']}) != len(paths):\n            raise ValueError(\'多个源文件的输出相对路径发生冲突\')\n        self.save(job)\n        self.queue.put(job)\n        return job\n\n    def worker(self):\n        while True:\n            job = self.queue.get()\n            try:\n                self.run(job)\n            except Exception as error:\n                job.update(status=\'failed\', error=str(error))\n                self.save(job)\n            finally:\n                self.queue.task_done()\n\n    def run(self, job):\n        job[\'status\'] = \'running\'\n        self.save(job)\n        for item in job[\'files\']:\n            source = Path(item[\'source\'])\n            destination = drive_path(str(Path(job[\'output_dir\']) / item.get(\'relative_output\', str(source.relative_to(DRIVE)))))\n            marker = destination / \'complete.json\'\n            try:\n                signature = file_signature(source, job[\'options\'])\n                if (not job[\'overwrite\'] or item.get(\'completed_signature\') == signature) and marker.is_file():\n                    previous = json.loads(marker.read_text())\n                    if previous.get(\'signature\') == signature and (destination / \'result.md\').is_file():\n                        item.update(status=\'skipped\', output=str(destination))\n                        self.save(job)\n                        continue\n                # An old completion marker must never validate a partially overwritten result.\n                marker.unlink(missing_ok=True)\n                item[\'status\'] = \'running\'\n                item.pop(\'error\', None)\n                self.save(job)\n                def progress(done, total):\n                    item.update(pages_done=done, pages_total=total)\n                    self.save(job)\n                conversion_opts = {**job[\'options\'],\n                                   \'_checkpoint_generation\': job[\'id\'] if job[\'overwrite\'] else \'shared\'}\n                result = self.engine.convert(source, destination, conversion_opts, progress)\n                if file_signature(source, job[\'options\']) != signature:\n                    raise ValueError(\'转换期间源文件发生变化；请恢复任务以重新处理\')\n                atomic_json(marker, dict(signature=signature, source=str(source), options=job[\'options\'], **result))\n                item.update(status=\'completed\', output=str(destination), completed_signature=signature, **result)\n            except Exception as error:\n                item.update(status=\'failed\', error=str(error))\n            self.save(job)\n        job.update(status=\'failed\' if any(f[\'status\'] == \'failed\' for f in job[\'files\']) else \'completed\',\n                   finished=time.time())\n        self.save(job)\n\n\ndef upload_chunk(body):\n    """Stage local input in a content-addressed Drive inbox, in bounded chunks."""\n    digest = body.get(\'sha256\', \'\')\n    name = body.get(\'name\', \'\')\n    if not re.fullmatch(r\'[0-9a-f]{64}\', digest) or Path(name).name != name or Path(name).suffix.lower() not in EXTENSIONS:\n        raise ValueError(\'非法上传名称或 SHA256\')\n    offset = int(body.get(\'offset\', 0))\n    chunk = base64.b64decode(body.get(\'data\', \'\'), validate=True)\n    if offset < 0 or len(chunk) > 512 * 1024:\n        raise ValueError(\'非法分块大小/偏移\')\n    folder = drive_path(str(DATA / \'uploads\' / digest))\n    folder.mkdir(parents=True, exist_ok=True)\n    destination = folder / name\n    temporary = folder / (name + \'.part\')\n    if destination.exists():\n        return dict(path=str(destination), complete=True)\n    if offset == 0:\n        temporary.write_bytes(chunk)\n    else:\n        if not temporary.exists() or temporary.stat().st_size != offset:\n            raise ValueError(\'上传偏移不匹配，请重试上传\')\n        with temporary.open(\'ab\') as stream:\n            stream.write(chunk)\n    if body.get(\'final\'):\n        hasher = hashlib.sha256()\n        with temporary.open(\'rb\') as stream:\n            for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n                hasher.update(block)\n        if hasher.hexdigest() != digest:\n            temporary.unlink()\n            raise ValueError(\'上传 SHA256 校验失败\')\n        os.replace(temporary, destination)\n    return dict(path=str(destination), complete=destination.exists())\n\n\ndef result_files(jobs, job_id, index, name=None, offset=0):\n    record = jobs.get(job_id)\n    item = record[\'files\'][index]\n    if item[\'status\'] not in (\'completed\', \'skipped\'):\n        raise ValueError(\'文件尚未完成转换\')\n    root = drive_path(item[\'output\'])\n    if name is None:\n        return dict(files=[str(p.relative_to(root)) for p in sorted(root.rglob(\'*\'))\n                           if p.is_file() and \'.checkpoints\' not in p.relative_to(root).parts\n                           and p.resolve().is_relative_to(root)])\n    target = (root / name).resolve()\n    if not target.is_relative_to(root) or not target.is_file() or offset < 0:\n        raise ValueError(\'非法结果文件路径/偏移\')\n    with target.open(\'rb\') as stream:\n        stream.seek(offset)\n        chunk = stream.read(512 * 1024)\n    return dict(data=base64.b64encode(chunk).decode(), next_offset=offset + len(chunk),\n                complete=offset + len(chunk) >= target.stat().st_size)\n\n\n_INLINE_MIME = {\n    \'.jpg\': \'image/jpeg\',\n    \'.jpeg\': \'image/jpeg\',\n    \'.png\': \'image/png\',\n    \'.webp\': \'image/webp\',\n    \'.gif\': \'image/gif\',\n    \'.md\': \'text/markdown; charset=utf-8\',\n    \'.txt\': \'text/plain; charset=utf-8\',\n    \'.json\': \'application/json; charset=utf-8\',\n}\n\n\ndef _job_file(jobs, job_id: str, index: int) -> tuple[dict, dict]:\n    if index < 0:\n        raise IndexError(\'file index out of range\')\n    record = jobs.get(job_id)\n    files = record.get(\'files\') or []\n    if index >= len(files):\n        raise IndexError(\'file index out of range\')\n    return record, files[index]\n\n\ndef _count_source_pages(source: Path) -> int | None:\n    if not source.is_file():\n        return None\n    if source.suffix.lower() == \'.pdf\':\n        try:\n            import fitz\n            with fitz.open(source) as pdf:\n                return len(pdf)\n        except Exception:\n            return None\n    try:\n        from PIL import Image\n        with Image.open(source) as img:\n            return getattr(img, \'n_frames\', 1)\n    except Exception:\n        return None\n\n\ndef render_source_page_png(jobs, job_id: str, index: int, page: int, dpi: int = 144) -> bytes:\n    if page < 1:\n        raise ValueError(\'页码必须 >= 1\')\n    dpi = max(72, min(200, int(dpi)))\n    _, item = _job_file(jobs, job_id, index)\n    source = drive_path(item[\'source\'])\n    if not source.is_file():\n        raise FileNotFoundError(\'原始文件不存在\')\n    if source.suffix.lower() == \'.pdf\':\n        import fitz\n        with fitz.open(source) as pdf:\n            if page > len(pdf):\n                raise ValueError(\'页码超出范围\')\n            return pdf[page - 1].get_pixmap(dpi=dpi, alpha=False).tobytes(\'png\')\n    from PIL import Image\n    buf = io.BytesIO()\n    with Image.open(source) as img:\n        total = getattr(img, \'n_frames\', 1)\n        if page > total:\n            raise ValueError(\'页码超出范围\')\n        if total > 1:\n            img.seek(page - 1)\n        img.convert(\'RGB\').save(buf, format=\'PNG\')\n    return buf.getvalue()\n\n\ndef file_preview(jobs, job_id: str, index: int) -> dict:\n    _, item = _job_file(jobs, job_id, index)\n    source_spec = item.get(\'source\') or \'\'\n    source = None\n    source_available = False\n    if source_spec:\n        try:\n            source = drive_path(source_spec)\n            source_available = source.is_file()\n        except ValueError:\n            source_available = False\n    markdown = \'\'\n    raw = \'\'\n    assets: list[str] = []\n    if item.get(\'status\') in (\'completed\', \'skipped\') and item.get(\'output\'):\n        try:\n            out_root = drive_path(item[\'output\'])\n            if out_root.is_dir():\n                md_path = (out_root / \'result.md\').resolve()\n                raw_path = (out_root / \'result.raw.txt\').resolve()\n                if md_path.is_file() and md_path.is_relative_to(out_root):\n                    markdown = md_path.read_text(encoding=\'utf-8\', errors=\'replace\')\n                if raw_path.is_file() and raw_path.is_relative_to(out_root):\n                    raw = raw_path.read_text(encoding=\'utf-8\', errors=\'replace\')\n                assets = [\n                    str(p.relative_to(out_root))\n                    for p in sorted(out_root.rglob(\'*\'))\n                    if p.is_file() and \'.checkpoints\' not in p.relative_to(out_root).parts\n                    and p.resolve().is_relative_to(out_root)\n                ]\n        except (OSError, ValueError):\n            pass\n    blocks = parse_raw_blocks(raw)\n    page_count = _count_source_pages(source) if (source is not None and source_available) else None\n    if page_count is None:\n        candidates = [item.get(\'pages\'), item.get(\'pages_total\'), max((b[\'page\'] for b in blocks), default=0)]\n        page_count = max(int(v or 0) for v in candidates)\n    return {\n        \'job_id\': job_id,\n        \'index\': index,\n        \'filename\': Path(source_spec).name if source_spec else (item.get(\'relative_output\') or f\'file-{index + 1}\'),\n        \'relative_output\': item.get(\'relative_output\'),\n        \'status\': item.get(\'status\'),\n        \'pages_done\': item.get(\'pages_done\'),\n        \'pages_total\': item.get(\'pages_total\'),\n        \'page_count\': page_count,\n        \'source_available\': source_available,\n        \'markdown\': markdown,\n        \'raw\': raw,\n        \'blocks\': blocks,\n        \'assets\': assets,\n        \'pages\': item.get(\'pages\'),\n        \'tokens\': item.get(\'tokens\'),\n        \'chunks\': item.get(\'chunks\'),\n        \'error\': item.get(\'error\'),\n    }\n\n\ndef read_result_asset(jobs, job_id: str, index: int, rel_path: str) -> tuple[bytes, str, bool]:\n    _, item = _job_file(jobs, job_id, index)\n    if item.get(\'status\') not in (\'completed\', \'skipped\') or not item.get(\'output\'):\n        raise ValueError(\'文件尚未完成转换\')\n    clean = urllib.parse.unquote(rel_path).lstrip(\'/\')\n    rel = Path(clean)\n    if not clean or rel.is_absolute() or \'..\' in rel.parts or \'.checkpoints\' in rel.parts:\n        raise ValueError(\'非法结果文件路径\')\n    root = drive_path(item[\'output\'])\n    target = (root / rel).resolve()\n    if not target.is_relative_to(root) or not target.is_file():\n        raise FileNotFoundError(\'结果文件不存在\')\n    mime = _INLINE_MIME.get(target.suffix.lower())\n    if mime is None:\n        return target.read_bytes(), \'application/octet-stream\', True\n    return target.read_bytes(), mime, False\n\n\ndef make_server(jobs, host: str = \'127.0.0.1\', port: int = 8000) -> ThreadingHTTPServer:\n    class Handler(BaseHTTPRequestHandler):\n        def log_message(self, format, *args):\n            return\n\n        def send_bytes(self, code: int, content_type: str, data: bytes, *, headers: dict | None = None):\n            self.send_response(code)\n            self.send_header(\'Content-Type\', content_type)\n            self.send_header(\'Content-Length\', str(len(data)))\n            self.send_header(\'X-Content-Type-Options\', \'nosniff\')\n            self.send_header(\'Referrer-Policy\', \'no-referrer\')\n            for k, v in (headers or {}).items():\n                self.send_header(k, v)\n            self.end_headers()\n            if self.command != \'HEAD\':\n                self.wfile.write(data)\n\n        def send_json(self, code, value):\n            raw = json.dumps(value, ensure_ascii=False).encode()\n            self.send_bytes(code, \'application/json; charset=utf-8\', raw)\n\n        def do_HEAD(self):\n            self.do_GET()\n\n        def do_GET(self):\n            try:\n                parsed = urllib.parse.urlsplit(self.path)\n                path = parsed.path\n                query = urllib.parse.parse_qs(parsed.query)\n                if path == \'/\':\n                    self.send_response(302)\n                    self.send_header(\'Location\', \'/demo\')\n                    self.send_header(\'Content-Length\', \'0\')\n                    self.send_header(\'Cache-Control\', \'no-store\')\n                    self.end_headers()\n                    return\n                if path == \'/demo\':\n                    nonce = secrets.token_urlsafe(16)\n                    return self.send_bytes(200, \'text/html; charset=utf-8\', render_demo_html(nonce), headers={\n                        \'Content-Security-Policy\': make_csp(nonce),\n                        \'Cache-Control\': \'no-store\',\n                    })\n                if path == \'/health\':\n                    return self.send_json(200, dict(status=\'ready\', model=\'baidu/Unlimited-OCR-2\'))\n                if path == \'/api/jobs\':\n                    return self.send_json(200, dict(jobs=jobs.list_jobs()))\n                m = re.fullmatch(r\'/api/jobs/([0-9a-f]{32})/files/([0-9]+)/preview\', path)\n                if m:\n                    return self.send_json(200, file_preview(jobs, m.group(1), int(m.group(2))))\n                m = re.fullmatch(r\'/api/jobs/([0-9a-f]{32})/files/([0-9]+)/pages/([0-9]+)\', path)\n                if m:\n                    dpi = int((query.get(\'dpi\') or [\'144\'])[0])\n                    png = render_source_page_png(jobs, m.group(1), int(m.group(2)), int(m.group(3)), dpi=dpi)\n                    return self.send_bytes(200, \'image/png\', png, headers={\'Cache-Control\': \'private, max-age=300\'})\n                m = re.fullmatch(r\'/api/jobs/([0-9a-f]{32})/files/([0-9]+)/asset/(.+)\', path)\n                if m:\n                    data, mime, force_dl = read_result_asset(jobs, m.group(1), int(m.group(2)), m.group(3))\n                    hdrs = {\'Cache-Control\': \'private, max-age=300\'}\n                    if force_dl or (query.get(\'download\') or [\'\'])[0] in (\'1\', \'true\'):\n                        fname = Path(urllib.parse.unquote(m.group(3))).name.replace(\'"\', \'_\')\n                        hdrs[\'Content-Disposition\'] = f\'attachment; filename="{fname}"\'\n                    return self.send_bytes(200, mime, data, headers=hdrs)\n                if path.startswith(\'/api/jobs/\'):\n                    return self.send_json(200, jobs.get(path.removeprefix(\'/api/jobs/\')))\n                self.send_json(404, dict(error=\'not found\'))\n            except (FileNotFoundError, IndexError):\n                self.send_json(404, dict(error=\'not found\'))\n            except ValueError as error:\n                self.send_json(400, dict(error=str(error)))\n\n        def do_POST(self):\n            try:\n                size = int(self.headers.get(\'Content-Length\', \'0\'))\n                if not 0 < size <= 1024 * 1024:\n                    raise ValueError(\'无效请求长度\')\n                body = json.loads(self.rfile.read(size))\n                if not isinstance(body, dict):\n                    raise ValueError(\'JSON body must be an object\')\n                if self.path == \'/api/upload\':\n                    return self.send_json(200, upload_chunk(body))\n                if self.path == \'/api/results\':\n                    return self.send_json(200, result_files(jobs, body[\'job_id\'], int(body[\'index\']),\n                        body.get(\'name\'), int(body.get(\'offset\', 0))))\n                if re.fullmatch(r\'/api/jobs/[0-9a-f]{32}/resume\', self.path):\n                    return self.send_json(202, jobs.resume(self.path.split(\'/\')[3]))\n                if self.path == \'/api/jobs\':\n                    return self.send_json(202, jobs.submit(body))\n                if self.path == \'/api/drive-list\':\n                    return self.send_json(200, dict(files=[str(p.relative_to(DRIVE)) for p in\n                        discover(body.get(\'inputs\'), body.get(\'recursive\', True))]))\n                self.send_json(404, dict(error=\'not found\'))\n            except (ValueError, TypeError, OSError, KeyError, IndexError) as error:\n                self.send_json(400, dict(error=str(error)))\n\n    return ThreadingHTTPServer((host, port), Handler)\n\n\ndef serve():\n    engine = Engine()\n    jobs = Jobs(engine)\n    make_server(jobs, \'127.0.0.1\', int(os.environ.get(\'CT_SERVICE_PORT\', \'8000\'))).serve_forever()\n\n\nif __name__ == \'__main__\':\n    serve()\n', encoding='utf-8')
def start_service():
    return ct_spawn("service", [OCR_PY, str(OCR_WORK / 'unlimited_ocr_app.py')], cwd=str(OCR_WORK),
                    env={"CT_SERVICE_PORT": str(SERVICE_PORT), "CT_DATA_DIR": DATA_DIR,
                         "CT_OCR_MODEL_DIR": str(OCR_MODEL),
                         "TRITON_CACHE_DIR": str(OCR_WORK / 'triton-cache'),
                         "TORCHINDUCTOR_CACHE_DIR": str(OCR_WORK / 'inductor-cache'),
                         "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})
PROCS["service"] = (start_service(), start_service)


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)

def start_caddy():
    caddy_bin = ct_ensure_caddy()
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output([caddy_bin, "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    ct_write_private_text(f"{WORK}/Caddyfile", caddyfile)
    return ct_spawn_marked("caddy", [caddy_bin, "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

CLOUDFLARED_VERSION = '2026.11.0'
CLOUDFLARED_SHA256 = {
    'amd64': '51f7c68b57ed7269164075d7e789efce5b59355982f06392deeb037d982a51fd',
    'arm64': 'abe54a006acf9faf2d19ee9d953cb0d0804e92f59c5382a162e57b84d086f56f',
}


def _write_private_text(path, content):
    path = Path(path)
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    os.chmod(path, 0o600)
    with os.fdopen(fd, 'w', encoding='utf-8') as handle:
        handle.write(content)
        handle.flush()
        os.fsync(handle.fileno())


def _touch_private_log(path):
    path = Path(path)
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    os.chmod(path, 0o600)
    os.close(fd)


def _file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            chunk = handle.read(65536)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def _check_cached_binary(destination, artifact, version, cache_dir):
    if not destination.exists():
        return None
    if not cache_dir:
        return 'hit'
    manifest = Path(cache_dir) / 'manifests' / f'{artifact}.json'
    if not manifest.is_file():
        return 'miss'
    try:
        meta = json.loads(manifest.read_text(encoding='utf-8'))
        if (
            meta.get('artifact') == artifact
            and meta.get('version') == version
            and meta.get('sha256') == _file_sha256(destination)
        ):
            return 'hit'
    except (OSError, ValueError):
        pass
    destination.unlink(missing_ok=True)
    return 'corrupt_rebuilt'


def _write_manifest(cache_dir, artifact, version, destination):
    if not cache_dir:
        return
    manifest_dir = Path(cache_dir) / 'manifests'
    manifest_dir.mkdir(parents=True, exist_ok=True)
    manifest = manifest_dir / f'{artifact}.json'
    tmp = manifest.with_name(f'.{artifact}.{os.getpid()}.tmp')
    payload = {
        'artifact': artifact,
        'version': version,
        'sha256': _file_sha256(destination),
        'bin_path': str(destination),
        'verified_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    tmp.write_text(json.dumps(payload, indent=2) + '\n', encoding='utf-8')
    os.replace(tmp, manifest)


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version, *, cache_dir=None):
    destination = Path(destination)
    cache_state = _check_cached_binary(destination, 'frpc', version, cache_dir)
    if cache_state == 'hit':
        if cache_dir:
            print(f'CT_CACHE_EVENT artifact=frpc status=hit', flush=True)
        return str(destination)
    status_label = cache_state or 'miss'
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            partial=destination.with_name(f'{destination.name}.{os.getpid()}.partial')
            try:
                with tar.extractfile(entry) as source, open(partial, 'wb') as out_bin:
                    shutil.copyfileobj(source, out_bin)
                partial.chmod(0o755)
                os.replace(partial, destination)
            finally:
                partial.unlink(missing_ok=True)
    _write_manifest(cache_dir, 'frpc', version, destination)
    if cache_dir:
        print(f'CT_CACHE_EVENT artifact=frpc status={status_label}', flush=True)
    return str(destination)


def install_cloudflared(destination, *, cache_dir=None, fetch_fn=None):
    destination = Path(destination)
    cf_arch = {'x86_64': 'amd64', 'aarch64': 'arm64'}.get(platform.machine(), 'amd64')
    url = f'https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-{cf_arch}'
    expected_sha = CLOUDFLARED_SHA256.get(cf_arch)
    if not expected_sha:
        raise ValueError(f'Unsupported cloudflared architecture: {platform.machine()}')
    if cache_dir:
        cache_state = _check_cached_binary(destination, 'cloudflared', CLOUDFLARED_VERSION, cache_dir)
        if cache_state == 'hit':
            print('CT_CACHE_EVENT artifact=cloudflared status=hit', flush=True)
            return str(destination)
        status_label = cache_state or 'miss'
        if destination.exists() and _file_sha256(destination) != expected_sha:
            destination.unlink(missing_ok=True)
            status_label = 'corrupt_rebuilt'
        destination.parent.mkdir(parents=True, exist_ok=True)
        partial = destination.with_name(f'{destination.name}.{os.getpid()}.partial')
        try:
            with urllib.request.urlopen(url, timeout=60) as src, open(partial, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            if _file_sha256(partial) != expected_sha:
                raise ValueError('cloudflared checksum verification failed')
            partial.chmod(0o755)
            os.replace(partial, destination)
        finally:
            partial.unlink(missing_ok=True)
        _write_manifest(cache_dir, 'cloudflared', CLOUDFLARED_VERSION, destination)
        print(f'CT_CACHE_EVENT artifact=cloudflared status={status_label}', flush=True)
        return str(destination)
    if fetch_fn is not None:
        return fetch_fn(str(destination), url)
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        partial = destination.with_name(f'{destination.name}.{os.getpid()}.partial')
        try:
            with urllib.request.urlopen(url, timeout=60) as src, open(partial, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            if _file_sha256(partial) != expected_sha:
                raise ValueError('cloudflared checksum verification failed')
            partial.chmod(0o755)
            os.replace(partial, destination)
        finally:
            partial.unlink(missing_ok=True)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx.get('WORK') or os.environ.get('CT_WORK_DIR') or '/content/ct')
    work.mkdir(parents=True, exist_ok=True)
    bin_dir_explicit = ctx.get('BIN_DIR') or os.environ.get('CT_BIN_DIR') or ''
    bin_dir = Path(bin_dir_explicit or '/usr/local/bin')
    cache_dir = ctx.get('CACHE_DIR') or os.environ.get('CT_CACHE_DIR') or ''
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if ctx.get('ct_stop_marked_proc') is not None:
        stop_fn = ctx['ct_stop_marked_proc']
        stop_fn('tunnel-process')
    elif marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        _touch_private_log(work/(name+'.log'))
        proc=ctx['ct_spawn'](name,args)
        _write_private_text(marker, json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        _write_private_text(ca_path, os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        _write_private_text(config_path, render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        frpc_name = f"frpc-{os.environ['CT_FRP_VERSION']}"
        frpc_dest = (bin_dir / frpc_name) if bin_dir_explicit else ((work / frpc_name) if (work / frpc_name).exists() else (bin_dir / frpc_name))
        binary=install_frpc(frpc_dest, os.environ['CT_FRP_VERSION'], cache_dir=cache_dir or None)
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        binary=install_cloudflared(bin_dir / 'cloudflared', cache_dir=cache_dir or None, fetch_fn=ctx.get('ct_fetch_binary'))
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see {work}/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Ready + keep-alive -- KEEP THIS CELL RUNNING
# The Colab session only lives while the `colab exec` connection is open and this cell blocks.
# The watchdog restarts any process from PROCS that died; a heartbeat line is printed every 4 min.
ct_wait_http(f"http://127.0.0.1:{SERVICE_PORT}/health", timeout=5400)
if AUTH == "caddy" and not CREDS_FROM_ENV:   # started from the Colab UI: show generated credentials
    print(f"CT_CREDENTIALS user={AUTH_USER} pass={AUTH_PASS}")
    print(f"API token: {API_TOKEN}")
for local in OCR_CACHE_PAIRS:
    try:
        save_cache(local, Path(DATA_DIR) / 'kernel-archives', OCR_CACHE_KEY)
    except (OSError, ValueError) as err:
        print(f'Cache archive save warning ({local.name}): {err}', flush=True)
print(f"{SERVICE} is up: {PUBLIC_URL}")
print("CT_READY", flush=True)

last_beat = time.time()
while True:
    time.sleep(30)
    for name, (proc, restart) in list(PROCS.items()):
        if proc.poll() is not None:
            print(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 15)}", flush=True)
            ct_stop_proc(proc)
            PROCS[name] = (restart(), restart)
    if time.time() - last_beat > 240:
        last_beat = time.time()
        print(f"[heartbeat] {time.strftime('%H:%M:%S')} " +
              " ".join(f"{n}={'up' if p.poll() is None else 'down'}" for n, (p, _) in PROCS.items()), flush=True)
